# A0 Full-Test Checkpoint Action Distribution

This notebook builds action-distribution plots from the checkpoint recorded in `outputs/full_test_eval/*.json`. It does **not** use the old mean of the last 5 logged W&B points. The full-test JSON selects the checkpoint and step; cached history supplies the nearest requested evaluation action metric at that step.

In [15]:
from pathlib import Path
import re
import sys

import pandas as pd


def find_task_dir(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "main.py").is_file() and candidate.name == "Topology_Task":
            return candidate
        task_dir = candidate / "Topology_Task"
        if (task_dir / "main.py").is_file():
            return task_dir
    raise RuntimeError("Could not locate Topology_Task/main.py")


TASK_DIR = find_task_dir()
HELPER_DIR = TASK_DIR / "analysis" / "metrics" / "helpers"
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

import action_distribution_metrics as adm

print("Task dir:", TASK_DIR)
print("Helper dir:", HELPER_DIR)

Task dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
Helper dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers


In [16]:
# Choose which cached run-data source to use for histories.
# Use "cpu", "gpu", or "all". Unsplit folders such as a0_hvg are kept as shared baselines.
RUN_DATA_SOURCE = "all"

# Keep this broad to include future a0_sparse16/a0_aib full-test JSONs when they exist.
FULL_TEST_CHECKPOINT_REGEX = r"^a0_"

# W&B histories do not always have an eval row exactly at checkpoint_global_step.
# "at_or_before" uses the latest eval metric at or before the checkpoint step and reports the delta.
# Other accepted values: "exact", "nearest".
STEP_MATCH_POLICY = "at_or_before"
MAX_STEP_DELTA = None  # set an integer number of env steps to reject distant cached eval points

CONFIG_FOLDERS_TO_LOAD = ["a0_hvg", "a0_sparse16", "a0_aib"]
ACTION_METRIC_SOURCE = "eval"
EVAL_METRIC_SPLIT = "test"
SAVE_FIGURES = True
SHOW_FIGURES = True

# Keep broad debug tables hidden by default.
SHOW_DIAGNOSTIC_TABLES = False

# Show the exact checkpoint files used before each checkpoint-aligned plot.
SHOW_CHECKPOINT_AUDIT_TABLES = True

## Full-Test / Checkpoint Coverage

In [17]:
full_test = adm.load_full_test_eval_results()
pattern = re.compile(FULL_TEST_CHECKPOINT_REGEX)
a0_full_test = full_test[
    full_test["run_like"].fillna("").astype(str).map(lambda value: bool(pattern.search(value)))
].copy()

if SHOW_DIAGNOSTIC_TABLES:
    print(f"A0 full-test eval JSON rows matching {FULL_TEST_CHECKPOINT_REGEX!r}: {len(a0_full_test)}")
    adm.print_full_test_eval_source_folders(a0_full_test)
    display(a0_full_test[[
        "run_like",
        "checkpoint_global_step",
        "survival_percent",
        "split",
        "eval_episodes",
        "local_checkpoint_exists",
        "local_checkpoint_path",
        "local_best_test_checkpoint_path",
        "path",
    ]].sort_values(["run_like", "checkpoint_global_step"]))


In [18]:
checkpoint_dir = TASK_DIR / "checkpoint" / "with_obs_stats"
local_best = pd.DataFrame(
    {
        "checkpoint_path": [str(path) for path in sorted(checkpoint_dir.glob("best_test_a0_*.tar"))]
    }
)
if not local_best.empty:
    local_best["run_like"] = local_best["checkpoint_path"].map(lambda value: Path(value).stem.removeprefix("best_test_"))
    full_test_run_likes = set(a0_full_test["run_like"].dropna().astype(str))
    missing_full_test = local_best[~local_best["run_like"].isin(full_test_run_likes)].copy()
else:
    missing_full_test = local_best

if SHOW_DIAGNOSTIC_TABLES:
    print(f"Local best_test_a0 checkpoints without matching full_test_eval JSON: {len(missing_full_test)}")
    display(missing_full_test)


## Load Cached Histories

In [19]:
ctx = adm.load_action_distribution_context(
    experiment_folders=CONFIG_FOLDERS_TO_LOAD,
    action_metric_source=ACTION_METRIC_SOURCE,
    eval_split=EVAL_METRIC_SPLIT,
    run_data_source=RUN_DATA_SOURCE,
    save_figures=SAVE_FIGURES,
    show_figures=SHOW_FIGURES,
)

if SHOW_DIAGNOSTIC_TABLES:
    display(ctx["coverage"].groupby(["experiment", "family_label"], dropna=False).agg(
        expected=("expected_run_name", "count"),
        cached=("cached", "sum"),
        seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
    ).reset_index())


Expected configs: 54
Cached expected runs: 93 / 93
[  1/93] loading a0_aib_00_flat_local_t020_s0


<action_distribution_metrics:configs_and_cache>:365: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.


[  2/93] loading a0_aib_00_flat_local_t020_s1
[  3/93] loading a0_aib_00_flat_local_t020_s2
[  4/93] loading a0_aib_01_flat_local_t010_s0
[  5/93] loading a0_aib_01_flat_local_t010_s1
[  6/93] loading a0_aib_01_flat_local_t010_s2
[  7/93] loading a0_aib_02_flat_local_t035_s0
[  8/93] loading a0_aib_02_flat_local_t035_s1
[  9/93] loading a0_aib_02_flat_local_t035_s2
[ 10/93] loading a0_aib_03_gate_hgreedy_sep_local_t020_s0
[ 11/93] loading a0_aib_03_gate_hgreedy_sep_local_t020_s1
[ 12/93] loading a0_aib_03_gate_hgreedy_sep_local_t020_s2
[ 13/93] loading a0_aib_04_flat_nonidle_t020_s0
[ 14/93] loading a0_aib_04_flat_nonidle_t020_s1
[ 15/93] loading a0_aib_04_flat_nonidle_t020_s2
[ 16/93] loading a0_aib_00_flat_local_t020_s0
[ 17/93] loading a0_aib_00_flat_local_t020_s1
[ 18/93] loading a0_aib_00_flat_local_t020_s2
[ 19/93] loading a0_aib_01_flat_local_t010_s0
[ 20/93] loading a0_aib_01_flat_local_t010_s1
[ 21/93] loading a0_aib_01_flat_local_t010_s2
[ 22/93] loading a0_aib_02_flat_local_

## A0 HVG Baseline Action-0 By Seed

This plot reads `best_test_a0_hvg_00_baseline_s*.tar`, extracts each checkpoint `global_step`, and shows the cached evaluation action-0 / do-nothing fraction at that checkpoint step for every agent and seed.


In [20]:
import plotly.express as px
import torch

A0_HVG_BASELINE_LABEL = "a0_hvg_00_baseline"
BASELINE_RUN_REGEX = rf"^{A0_HVG_BASELINE_LABEL}_s[0-2]$"
BASELINE_AGENTS = ["agent_0", "agent_1", "agent_2"]
BEST_TEST_CHECKPOINT_DIR = TASK_DIR / "checkpoint" / "with_obs_stats"


def _seed_from_run_name(run_name):
    match = re.search(r"_s(\d+)$", str(run_name))
    return int(match.group(1)) if match else None


def _load_checkpoint_global_step(path):
    record = torch.load(path, map_location="cpu", weights_only=False)
    return int(record.get("global_step"))


def _checkpoint_audit_table(seed_rows, *, labels=None, plot_group=None):
    base_columns = [
        "condition_label",
        "run_name",
        "seed",
        "checkpoint_file",
        "checkpoint_global_step",
        "selected_metric_step",
        "step_delta",
        "step_match_policy",
        "checkpoint_path",
    ]
    if seed_rows is None or seed_rows.empty:
        return pd.DataFrame(columns=base_columns)

    audit = seed_rows.copy()
    if labels is not None and "condition_label" in audit.columns:
        audit = audit[audit["condition_label"].astype(str).isin([str(label) for label in labels])].copy()
    if plot_group is not None and "plot_group" in audit.columns:
        baseline_mask = audit.get("condition_label", pd.Series(index=audit.index, dtype=object)).astype(str).eq(A0_HVG_BASELINE_LABEL)
        audit = audit[audit["plot_group"].astype(str).eq(str(plot_group)) | baseline_mask].copy()

    if audit.empty:
        return pd.DataFrame(columns=base_columns)

    if "checkpoint_path" not in audit.columns:
        audit["checkpoint_path"] = ""
    audit["checkpoint_path"] = audit["checkpoint_path"].fillna("").astype(str)
    audit["checkpoint_file"] = audit["checkpoint_path"].map(lambda value: Path(value).name if value else "")

    optional_defaults = {
        "condition_label": "",
        "run_name": "",
        "seed": pd.NA,
        "checkpoint_global_step": pd.NA,
        "selected_metric_step": pd.NA,
        "step_delta": pd.NA,
        "step_match_policy": "",
    }
    for column, default in optional_defaults.items():
        if column not in audit.columns:
            audit[column] = default

    key_columns = [
        "condition_label",
        "run_name",
        "seed",
        "checkpoint_file",
        "checkpoint_global_step",
        "selected_metric_step",
        "step_delta",
        "step_match_policy",
        "checkpoint_path",
    ]
    result = audit[key_columns].drop_duplicates().copy()

    if {"agent", "action0_fraction"}.issubset(audit.columns):
        seed_value_table = (
            audit
            .pivot_table(
                index=key_columns,
                columns="agent",
                values="action0_fraction",
                aggfunc="first",
                observed=True,
            )
            .reset_index()
        )
        seed_value_table = seed_value_table.rename(
            columns={
                agent: f"{agent}_plotted_action0_fraction"
                for agent in seed_value_table.columns
                if str(agent).startswith("agent_")
            }
        )
        result = result.merge(seed_value_table, on=key_columns, how="left")

        mean_index_columns = ["condition_label"]
        if "plot_group" in audit.columns:
            mean_index_columns.append("plot_group")
        mean_value_table = (
            audit
            .pivot_table(
                index=mean_index_columns,
                columns="agent",
                values="action0_fraction",
                aggfunc="mean",
                observed=True,
            )
            .reset_index()
        )
        mean_value_table = mean_value_table.rename(
            columns={
                agent: f"{agent}_mean_plotted_action0_fraction"
                for agent in mean_value_table.columns
                if str(agent).startswith("agent_")
            }
        )
        join_columns = [column for column in mean_index_columns if column in result.columns]
        if join_columns:
            if "plot_group" not in result.columns and "plot_group" in mean_value_table.columns:
                mean_value_table = mean_value_table.drop(columns=["plot_group"])
            result = result.merge(mean_value_table, on=join_columns, how="left")

    sort_columns = [column for column in ["condition_label", "run_name", "seed", "checkpoint_file"] if column in result.columns]
    return result.sort_values(sort_columns, kind="stable").reset_index(drop=True)


def _display_checkpoint_audit(seed_rows, *, title, labels=None, plot_group=None):
    audit = _checkpoint_audit_table(seed_rows, labels=labels, plot_group=plot_group)
    if SHOW_CHECKPOINT_AUDIT_TABLES:
        print(f"Checkpoint files used for: {title}")
        display(audit)
    return audit


history = ctx["history_wide"].copy()
selected_runs = ctx["selected_runs"].copy()
baseline_runs = selected_runs[
    selected_runs["run_name"].fillna("").astype(str).str.match(BASELINE_RUN_REGEX)
].sort_values("run_name")

baseline_action0_rows = []
baseline_missing_rows = []
for run in baseline_runs.to_dict("records"):
    run_name = str(run["run_name"])
    run_id = str(run["run_id"])
    seed = _seed_from_run_name(run_name)
    checkpoint_path = BEST_TEST_CHECKPOINT_DIR / f"best_test_{run_name}.tar"
    if not checkpoint_path.exists():
        baseline_missing_rows.append({
            "run_name": run_name,
            "seed": seed,
            "reason": "missing_best_test_checkpoint",
            "checkpoint_path": str(checkpoint_path),
        })
        continue

    try:
        checkpoint_step = _load_checkpoint_global_step(checkpoint_path)
    except Exception as exc:
        baseline_missing_rows.append({
            "run_name": run_name,
            "seed": seed,
            "reason": f"checkpoint_load_failed: {type(exc).__name__}: {exc}",
            "checkpoint_path": str(checkpoint_path),
        })
        continue

    run_history = history[history["run_id"].astype(str).eq(run_id)].sort_values("_step").copy()
    if run_history.empty:
        baseline_missing_rows.append({
            "run_name": run_name,
            "seed": seed,
            "reason": "missing_cached_history",
            "checkpoint_path": str(checkpoint_path),
            "checkpoint_global_step": checkpoint_step,
        })
        continue

    for agent in BASELINE_AGENTS:
        values, metric_column, inverted = adm._action0_series_for_checkpoint_agent(
            run_history,
            str(EVAL_METRIC_SPLIT),
            agent,
        )
        if values is None:
            baseline_missing_rows.append({
                "run_name": run_name,
                "seed": seed,
                "agent": agent,
                "reason": "missing_agent_action0_metric",
                "checkpoint_path": str(checkpoint_path),
                "checkpoint_global_step": checkpoint_step,
            })
            continue

        selected_point = adm._select_checkpoint_metric_point(
            run_history,
            values,
            checkpoint_step,
            step_policy=STEP_MATCH_POLICY,
        )
        if selected_point is None:
            baseline_missing_rows.append({
                "run_name": run_name,
                "seed": seed,
                "agent": agent,
                "reason": "no_metric_point_at_policy_step",
                "checkpoint_path": str(checkpoint_path),
                "checkpoint_global_step": checkpoint_step,
                "metric_column": metric_column,
            })
            continue

        point, match_policy = selected_point
        selected_step = float(point["_step"])
        step_delta = selected_step - float(checkpoint_step)
        if MAX_STEP_DELTA is not None and abs(step_delta) > float(MAX_STEP_DELTA):
            baseline_missing_rows.append({
                "run_name": run_name,
                "seed": seed,
                "agent": agent,
                "reason": "step_delta_exceeds_max",
                "checkpoint_path": str(checkpoint_path),
                "checkpoint_global_step": checkpoint_step,
                "selected_metric_step": selected_step,
                "step_delta": step_delta,
                "metric_column": metric_column,
            })
            continue

        baseline_action0_rows.append({
            "run_name": run_name,
            "run_id": run_id,
            "seed": seed,
            "seed_label": f"seed {seed}",
            "agent": agent,
            "action0_fraction": float(point["value"]),
            "metric_column": metric_column,
            "metric_inverted_from_nonidle": bool(inverted),
            "checkpoint_global_step": checkpoint_step,
            "checkpoint_step_millions": checkpoint_step / 1_000_000,
            "selected_metric_step": selected_step,
            "selected_metric_step_millions": selected_step / 1_000_000,
            "step_delta": step_delta,
            "step_match_policy": match_policy,
            "checkpoint_path": str(checkpoint_path),
        })

baseline_action0_by_seed = pd.DataFrame(baseline_action0_rows).sort_values(["agent", "seed"])
baseline_action0_missing = pd.DataFrame(baseline_missing_rows)

print(f"Best-test checkpoint-aligned action-0 rows for a0_hvg_00_baseline: {len(baseline_action0_by_seed)}")
if SHOW_DIAGNOSTIC_TABLES:
    display(baseline_action0_by_seed)
    if not baseline_action0_missing.empty:
        print("Missing / rejected baseline checkpoint-aligned rows:")
        display(baseline_action0_missing)
elif not baseline_action0_missing.empty:
    print(f"Baseline missing/rejected rows hidden: {len(baseline_action0_missing)}. Set SHOW_DIAGNOSTIC_TABLES=True to inspect them.")

plot_data = baseline_action0_by_seed.dropna(subset=["action0_fraction"]).copy()
if plot_data.empty:
    print("No checkpoint-aligned action-0 metric rows found for a0_hvg_00_baseline_s0/s1/s2.")
else:
    plot_data["seed_label"] = pd.Categorical(
        plot_data["seed_label"],
        categories=["seed 0", "seed 1", "seed 2"],
        ordered=True,
    )
    plot_data["agent"] = pd.Categorical(
        plot_data["agent"],
        categories=BASELINE_AGENTS,
        ordered=True,
    )
    baseline_action0_by_seed_audit = _display_checkpoint_audit(
        plot_data,
        title=f"A0 HVG {A0_HVG_BASELINE_LABEL}: action 0 / do-nothing usage at best-test checkpoint by seed",
    )

    fig_a0_hvg_baseline_action0_by_seed = px.bar(
        plot_data,
        x="agent",
        y="action0_fraction",
        color="seed_label",
        barmode="group",
        text=plot_data["action0_fraction"].map(lambda value: f"{value:.2f}"),
        hover_data={
            "run_name": True,
            "seed": True,
            "agent": True,
            "action0_fraction": ":.4f",
            "checkpoint_step_millions": ":.2f",
            "selected_metric_step_millions": ":.2f",
            "step_delta": ":.0f",
            "step_match_policy": True,
            "metric_column": True,
        },
        labels={
            "agent": "agent",
            "action0_fraction": "action 0 / do-nothing fraction",
            "seed_label": "seed",
        },
        title=(
            f"A0 HVG {A0_HVG_BASELINE_LABEL}: action 0 / do-nothing usage at best-test checkpoint "
            f"({STEP_MATCH_POLICY} metric match)"
        ),
    )
    fig_a0_hvg_baseline_action0_by_seed.update_yaxes(range=[0, 1])
    fig_a0_hvg_baseline_action0_by_seed.update_traces(textposition="outside", cliponaxis=False)
    fig_a0_hvg_baseline_action0_by_seed.update_layout(
        template="plotly_white",
        height=560,
        width=1050,
        bargap=0.20,
        bargroupgap=0.06,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "right", "x": 1},
        margin={"l": 70, "r": 40, "t": 110, "b": 70},
    )
    if SAVE_FIGURES:
        save_path = ctx["FIG_DIR"] / "a0_hvg_00_baseline_best_test_checkpoint_action0_by_agent_seed.html"
        fig_a0_hvg_baseline_action0_by_seed.write_html(save_path, include_plotlyjs="cdn")
        print(f"Saved: {save_path}")
    if SHOW_FIGURES:
        fig_a0_hvg_baseline_action0_by_seed.show()


Best-test checkpoint-aligned action-0 rows for a0_hvg_00_baseline: 9
Checkpoint files used for: A0 HVG a0_hvg_00_baseline: action 0 / do-nothing usage at best-test checkpoint by seed


,condition_label,run_name,seed,checkpoint_file,checkpoint_global_step,selected_metric_step,step_delta,step_match_policy,checkpoint_path,agent_0_plotted_action0_fraction,agent_1_plotted_action0_fraction,agent_2_plotted_action0_fraction,agent_0_mean_plotted_action0_fraction,agent_1_mean_plotted_action0_fraction,agent_2_mean_plotted_action0_fraction
0,,a0_hvg_00_baseline_s0,0,best_test_a0_hvg_00_baseline_s0.tar,13600000,13600000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.300781,0.145709,0.305370,0.555357,0.384102,0.368167
1,,a0_hvg_00_baseline_s1,1,best_test_a0_hvg_00_baseline_s1.tar,13040000,13040000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.450236,0.489993,0.469506,0.555357,0.384102,0.368167
2,,a0_hvg_00_baseline_s2,2,best_test_a0_hvg_00_baseline_s2.tar,13760000,13760000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.915055,0.516605,0.329625,0.555357,0.384102,0.368167


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/a0_hvg_00_baseline_best_test_checkpoint_action0_by_agent_seed.html


## A0 HVG Baseline Seed Average

This computes the mean action-0 / do-nothing fraction across the three `a0_hvg_00_baseline` seeds, using the same best-test checkpoint-aligned rows from the previous cell.


In [21]:
import plotly.graph_objects as go

seed_average_source = baseline_action0_by_seed.dropna(subset=["action0_fraction"]).copy()

if seed_average_source.empty:
    print("No baseline action-0 rows available to average across seeds.")
    baseline_action0_seed_average = pd.DataFrame()
else:
    baseline_action0_seed_average = (
        seed_average_source
        .groupby("agent", observed=True, as_index=False)
        .agg(
            mean_action0_fraction=("action0_fraction", "mean"),
            std_action0_fraction=("action0_fraction", "std"),
            min_action0_fraction=("action0_fraction", "min"),
            max_action0_fraction=("action0_fraction", "max"),
            n_seeds=("seed", "nunique"),
            seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
            runs=("run_name", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
            checkpoint_steps=("checkpoint_global_step", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
            selected_metric_steps=("selected_metric_step", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
        )
        .sort_values("agent")
    )
    baseline_action0_seed_average["std_action0_fraction"] = baseline_action0_seed_average["std_action0_fraction"].fillna(0.0)

    if SHOW_DIAGNOSTIC_TABLES:
        print("Best-test checkpoint-aligned action-0 average over seeds for a0_hvg_00_baseline:")
        display(baseline_action0_seed_average)

    baseline_action0_seed_average_audit = _display_checkpoint_audit(
        seed_average_source,
        title=f"A0 HVG {A0_HVG_BASELINE_LABEL}: mean action 0 / do-nothing usage over seeds at best-test checkpoints",
    )

    fig_a0_hvg_baseline_action0_seed_average = px.bar(
        baseline_action0_seed_average,
        x="agent",
        y="mean_action0_fraction",
        error_y="std_action0_fraction",
        text=baseline_action0_seed_average["mean_action0_fraction"].map(lambda value: f"{value:.2f}"),
        hover_data={
            "mean_action0_fraction": ":.4f",
            "std_action0_fraction": ":.4f",
            "min_action0_fraction": ":.4f",
            "max_action0_fraction": ":.4f",
            "n_seeds": True,
            "seeds": True,
            "checkpoint_steps": True,
            "selected_metric_steps": True,
        },
        labels={
            "agent": "agent",
            "mean_action0_fraction": "mean action 0 / do-nothing fraction",
        },
        title=f"A0 {A0_HVG_BASELINE_LABEL}: mean action 0 / do-nothing usage over seeds at best-test checkpoints",
    )
    for seed, seed_data in seed_average_source.sort_values(["seed", "agent"]).groupby("seed", sort=True):
        fig_a0_hvg_baseline_action0_seed_average.add_trace(
            go.Scatter(
                x=seed_data["agent"],
                y=seed_data["action0_fraction"],
                mode="markers",
                name=f"seed {int(seed)}",
                marker={"size": 10, "symbol": "circle", "line": {"color": "white", "width": 1}},
                customdata=seed_data[["run_name", "checkpoint_global_step", "selected_metric_step", "step_delta"]],
                hovertemplate=(
                    "run=%{customdata[0]}<br>"
                    "agent=%{x}<br>"
                    "seed action0=%{y:.4f}<br>"
                    "checkpoint_step=%{customdata[1]}<br>"
                    "selected_metric_step=%{customdata[2]}<br>"
                    "step_delta=%{customdata[3]}<extra></extra>"
                ),
            )
        )

    fig_a0_hvg_baseline_action0_seed_average.update_yaxes(range=[0, 1])
    fig_a0_hvg_baseline_action0_seed_average.update_traces(textposition="outside", selector={"type": "bar"}, cliponaxis=False)
    fig_a0_hvg_baseline_action0_seed_average.update_layout(
        template="plotly_white",
        height=560,
        width=1050,
        bargap=0.35,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "right", "x": 1},
        margin={"l": 70, "r": 40, "t": 110, "b": 70},
    )
    if SAVE_FIGURES:
        save_path = ctx["FIG_DIR"] / "a0_hvg_00_baseline_best_test_checkpoint_action0_seed_average.html"
        fig_a0_hvg_baseline_action0_seed_average.write_html(save_path, include_plotlyjs="cdn")
        print(f"Saved: {save_path}")
    if SHOW_FIGURES:
        fig_a0_hvg_baseline_action0_seed_average.show()


Checkpoint files used for: A0 HVG a0_hvg_00_baseline: mean action 0 / do-nothing usage over seeds at best-test checkpoints


,condition_label,run_name,seed,checkpoint_file,checkpoint_global_step,selected_metric_step,step_delta,step_match_policy,checkpoint_path,agent_0_plotted_action0_fraction,agent_1_plotted_action0_fraction,agent_2_plotted_action0_fraction,agent_0_mean_plotted_action0_fraction,agent_1_mean_plotted_action0_fraction,agent_2_mean_plotted_action0_fraction
0,,a0_hvg_00_baseline_s0,0,best_test_a0_hvg_00_baseline_s0.tar,13600000,13600000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.300781,0.145709,0.305370,0.555357,0.384102,0.368167
1,,a0_hvg_00_baseline_s1,1,best_test_a0_hvg_00_baseline_s1.tar,13040000,13040000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.450236,0.489993,0.469506,0.555357,0.384102,0.368167
2,,a0_hvg_00_baseline_s2,2,best_test_a0_hvg_00_baseline_s2.tar,13760000,13760000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.915055,0.516605,0.329625,0.555357,0.384102,0.368167


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/a0_hvg_00_baseline_best_test_checkpoint_action0_seed_average.html


## A0 HVG Checkpoint-Aligned Comparison Plots

These two plots use the `best_test_<run>.tar` checkpoint for each seed, then read the action-0 / do-nothing fraction from the cached evaluation metric at the checkpoint-aligned step. Bars show the mean over seeds; dots show the actual seed values. No uncertainty bars are drawn.


In [22]:
import re

import numpy as np
import plotly.graph_objects as go
import torch

BEST_TEST_CHECKPOINT_DIR = TASK_DIR / "checkpoint" / "with_obs_stats"


def _seed_from_run_name(run_name):
    match = re.search(r"_s(\d+)$", str(run_name))
    return int(match.group(1)) if match else None


def _load_checkpoint_global_step(path):
    record = torch.load(path, map_location="cpu", weights_only=False)
    return int(record.get("global_step"))


A0_HVG_CHECKPOINT_VARIANTS = [
    {
        "family": "a0_hvg_00_baseline",
        "label": A0_HVG_BASELINE_LABEL,
        "plot_group": "heuristic",
        "order": 0,
    },
    {
        "family": "a0_hvg_01_eval_rho090",
        "label": "global rho heuristic",
        "plot_group": "heuristic",
        "order": 1,
    },
    {
        "family": "a0_hvg_04_eval_local_rho090",
        "label": "local rho heuristic",
        "plot_group": "heuristic",
        "order": 2,
    },
    {
        "family": "a0_hvg_02_gate_final_map",
        "label": "gate final-action MAP",
        "plot_group": "gate",
        "order": 1,
    },
    {
        "family": "a0_hvg_03_gate_hierarchical",
        "label": "gate hierarchical greedy",
        "plot_group": "gate",
        "order": 2,
    },
]
A0_HVG_CHECKPOINT_AGENTS = ["agent_0", "agent_1", "agent_2"]
A0_HVG_VARIANT_COLORS = {
    A0_HVG_BASELINE_LABEL: "#1f77b4",
    "global rho heuristic": "#ff7f0e",
    "local rho heuristic": "#2ca02c",
    "gate final-action MAP": "#9467bd",
    "gate hierarchical greedy": "#d62728",
}
A0_AGENT_PATTERNS = {
    "agent_0": "",
    "agent_1": "/",
    "agent_2": "x",
}


def _collect_a0_hvg_best_test_checkpoint_action0_rows(variant_specs):
    history = ctx["history_wide"].copy()
    selected_runs = ctx["selected_runs"].copy()
    rows = []
    missing = []

    for spec in variant_specs:
        family = spec["family"]
        family_runs = selected_runs[
            selected_runs["run_name"].fillna("").astype(str).str.match(rf"^{re.escape(family)}_s[0-2]$")
        ].sort_values("run_name")
        if family_runs.empty:
            missing.append({"family": family, "reason": "missing_cached_runs"})
            continue

        for run in family_runs.to_dict("records"):
            run_name = str(run["run_name"])
            run_id = str(run["run_id"])
            seed = _seed_from_run_name(run_name)
            checkpoint_path = BEST_TEST_CHECKPOINT_DIR / f"best_test_{run_name}.tar"
            if not checkpoint_path.exists():
                missing.append({
                    "run_name": run_name,
                    "family": family,
                    "seed": seed,
                    "reason": "missing_best_test_checkpoint",
                    "checkpoint_path": str(checkpoint_path),
                })
                continue

            try:
                checkpoint_step = _load_checkpoint_global_step(checkpoint_path)
            except Exception as exc:
                missing.append({
                    "run_name": run_name,
                    "family": family,
                    "seed": seed,
                    "reason": f"checkpoint_load_failed: {type(exc).__name__}: {exc}",
                    "checkpoint_path": str(checkpoint_path),
                })
                continue

            run_history = history[history["run_id"].astype(str).eq(run_id)].sort_values("_step").copy()
            if run_history.empty:
                missing.append({
                    "run_name": run_name,
                    "family": family,
                    "seed": seed,
                    "reason": "missing_cached_history",
                    "checkpoint_path": str(checkpoint_path),
                    "checkpoint_global_step": checkpoint_step,
                })
                continue

            for agent in A0_HVG_CHECKPOINT_AGENTS:
                values, metric_column, inverted = adm._action0_series_for_checkpoint_agent(
                    run_history,
                    str(EVAL_METRIC_SPLIT),
                    agent,
                )
                if values is None:
                    missing.append({
                        "run_name": run_name,
                        "family": family,
                        "seed": seed,
                        "agent": agent,
                        "reason": "missing_agent_action0_metric",
                        "checkpoint_path": str(checkpoint_path),
                        "checkpoint_global_step": checkpoint_step,
                    })
                    continue

                selected_point = adm._select_checkpoint_metric_point(
                    run_history,
                    values,
                    checkpoint_step,
                    step_policy=STEP_MATCH_POLICY,
                )
                if selected_point is None:
                    missing.append({
                        "run_name": run_name,
                        "family": family,
                        "seed": seed,
                        "agent": agent,
                        "reason": "no_metric_point_at_policy_step",
                        "checkpoint_path": str(checkpoint_path),
                        "checkpoint_global_step": checkpoint_step,
                        "metric_column": metric_column,
                    })
                    continue

                point, match_policy = selected_point
                selected_step = float(point["_step"])
                step_delta = selected_step - float(checkpoint_step)
                if MAX_STEP_DELTA is not None and abs(step_delta) > float(MAX_STEP_DELTA):
                    missing.append({
                        "run_name": run_name,
                        "family": family,
                        "seed": seed,
                        "agent": agent,
                        "reason": "step_delta_exceeds_max",
                        "checkpoint_path": str(checkpoint_path),
                        "checkpoint_global_step": checkpoint_step,
                        "selected_metric_step": selected_step,
                        "step_delta": step_delta,
                        "metric_column": metric_column,
                    })
                    continue

                rows.append({
                    "family": family,
                    "condition_label": spec["label"],
                    "plot_group": spec["plot_group"],
                    "order": spec["order"],
                    "run_name": run_name,
                    "run_id": run_id,
                    "seed": seed,
                    "agent": agent,
                    "action0_fraction": float(point["value"]),
                    "checkpoint_global_step": checkpoint_step,
                    "selected_metric_step": selected_step,
                    "step_delta": step_delta,
                    "step_match_policy": match_policy,
                    "metric_column": metric_column,
                    "metric_inverted": bool(inverted),
                    "checkpoint_path": str(checkpoint_path),
                })

    return pd.DataFrame(rows), pd.DataFrame(missing)


def _checkpoint_action0_summary(seed_rows):
    if seed_rows.empty:
        return pd.DataFrame()
    return (
        seed_rows
        .groupby(["plot_group", "condition_label", "order", "agent"], observed=True, as_index=False)
        .agg(
            mean_action0_fraction=("action0_fraction", "mean"),
            n_seeds=("seed", "nunique"),
            seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
            runs=("run_name", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
            checkpoint_steps=("checkpoint_global_step", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
            selected_metric_steps=("selected_metric_step", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
        )
        .sort_values(["plot_group", "order", "agent"])
    )


def _plot_checkpoint_action0_grouped_bars(seed_rows, *, plot_group, labels, title, save_name):
    plot_seed_rows = seed_rows[
        seed_rows["condition_label"].isin(labels)
        & (seed_rows["plot_group"].eq(plot_group) | seed_rows["condition_label"].eq(A0_HVG_BASELINE_LABEL))
    ].copy()
    plot_seed_rows["plot_group"] = plot_group
    if plot_seed_rows.empty:
        print(f"No checkpoint-aligned action-0 rows available for {plot_group}.")
        return None, pd.DataFrame()

    plot_seed_rows["condition_label"] = pd.Categorical(plot_seed_rows["condition_label"], categories=labels, ordered=True)
    plot_seed_rows["agent"] = pd.Categorical(plot_seed_rows["agent"].astype(str), categories=A0_HVG_CHECKPOINT_AGENTS, ordered=True)
    summary = _checkpoint_action0_summary(plot_seed_rows)
    summary["condition_label"] = pd.Categorical(summary["condition_label"], categories=labels, ordered=True)
    summary["agent"] = pd.Categorical(summary["agent"].astype(str), categories=A0_HVG_CHECKPOINT_AGENTS, ordered=True)
    summary = summary.sort_values(["condition_label", "agent"])

    audit = _display_checkpoint_audit(
        plot_seed_rows,
        title=title,
        labels=labels,
        plot_group=plot_group,
    )

    if SHOW_DIAGNOSTIC_TABLES:
        print(title)
        display(summary)

    x_index = {label: idx for idx, label in enumerate(labels)}
    agent_width = 0.22
    agent_offsets = {
        agent: (idx - (len(A0_HVG_CHECKPOINT_AGENTS) - 1) / 2.0) * agent_width
        for idx, agent in enumerate(A0_HVG_CHECKPOINT_AGENTS)
    }
    bar_width = agent_width * 0.86

    fig = go.Figure()
    for label in labels:
        label_summary = summary[summary["condition_label"].astype(str).eq(label)].set_index("agent")
        x_values = []
        y_values = []
        text_values = []
        customdata = []
        pattern_shapes = []
        hover_agents = []
        for agent in A0_HVG_CHECKPOINT_AGENTS:
            x_values.append(x_index[label] + agent_offsets[agent])
            pattern_shapes.append(A0_AGENT_PATTERNS[agent])
            hover_agents.append(agent)
            if agent in label_summary.index:
                row = label_summary.loc[agent]
                y = float(row["mean_action0_fraction"])
                y_values.append(y)
                text_values.append(f"{y:.2f}")
                customdata.append([
                    agent,
                    int(row["n_seeds"]),
                    str(row["seeds"]),
                    str(row["checkpoint_steps"]),
                    str(row["selected_metric_steps"]),
                    str(row["runs"]),
                ])
            else:
                y_values.append(None)
                text_values.append("")
                customdata.append([agent, 0, "[]", "[]", "[]", "[]"])

        fig.add_trace(
            go.Bar(
                x=x_values,
                y=y_values,
                width=bar_width,
                name=label,
                legendgroup=label,
                marker={
                    "color": A0_HVG_VARIANT_COLORS[label],
                    "line": {"color": "rgba(0,0,0,0.35)", "width": 0.8},
                    "pattern": {"shape": pattern_shapes, "fgcolor": "rgba(255,255,255,0.75)", "size": 8},
                },
                text=text_values,
                textposition="outside",
                cliponaxis=False,
                customdata=np.array(customdata, dtype=object),
                hovertemplate=(
                    f"run type={label}<br>"
                    "agent=%{customdata[0]}<br>"
                    "mean action-0 fraction=%{y:.4f}<br>"
                    "n_seeds=%{customdata[1]}<br>"
                    "seeds=%{customdata[2]}<br>"
                    "checkpoint_steps=%{customdata[3]}<br>"
                    "selected_metric_steps=%{customdata[4]}<br>"
                    "runs=%{customdata[5]}<extra></extra>"
                ),
                hovertext=hover_agents,
            )
        )

    seed_marker_symbols = {0: "circle", 1: "diamond", 2: "square"}
    for (label, agent), group in plot_seed_rows.groupby(["condition_label", "agent"], observed=True, sort=False):
        label = str(label)
        agent = str(agent)
        group = group.sort_values("seed").copy()
        if group.empty:
            continue
        jitter = np.linspace(-bar_width * 0.22, bar_width * 0.22, len(group)) if len(group) > 1 else np.array([0.0])
        x_center = x_index[label] + agent_offsets[agent]
        fig.add_trace(
            go.Scatter(
                x=x_center + jitter,
                y=group["action0_fraction"],
                mode="markers",
                name=f"{label} {agent} seeds",
                legendgroup=label,
                showlegend=False,
                marker={
                    "color": A0_HVG_VARIANT_COLORS[label],
                    "size": 11,
                    "symbol": [seed_marker_symbols.get(int(seed), "circle") for seed in group["seed"]],
                    "line": {"color": "white", "width": 1.2},
                    "opacity": 0.95,
                },
                customdata=group[[
                    "run_name",
                    "seed",
                    "checkpoint_global_step",
                    "selected_metric_step",
                    "step_delta",
                    "metric_column",
                ]].to_numpy(dtype=object),
                hovertemplate=(
                    "run=%{customdata[0]}<br>"
                    "seed=%{customdata[1]}<br>"
                    f"agent={agent}<br>"
                    f"run type={label}<br>"
                    "action-0 fraction=%{y:.4f}<br>"
                    "checkpoint_step=%{customdata[2]}<br>"
                    "selected_metric_step=%{customdata[3]}<br>"
                    "step_delta=%{customdata[4]}<br>"
                    "metric=%{customdata[5]}<extra></extra>"
                ),
            )
        )

    fig.update_xaxes(
        tickmode="array",
        tickvals=list(range(len(labels))),
        ticktext=labels,
        title_text="run type",
    )
    fig.update_yaxes(range=[0, 1], title_text="action-0 / do-nothing fraction")
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=620,
        width=1250,
        bargap=0.20,
        legend={
            "title": {"text": "run variant"},
            "orientation": "v",
            "yanchor": "top",
            "y": 1,
            "xanchor": "left",
            "x": 1.01,
            "font": {"size": 15},
        },
        margin={"l": 75, "r": 180, "t": 95, "b": 120},
    )
    fig.add_annotation(
        text="Colors match episodic-survival run variants. Bar patterns: agent_0=solid, agent_1=slash, agent_2=cross. Dots: s0=circle, s1=diamond, s2=square.",
        xref="paper",
        yref="paper",
        x=0,
        y=-0.20,
        showarrow=False,
        align="left",
        font={"size": 12, "color": "#555"},
    )

    if SAVE_FIGURES:
        save_path = ctx["FIG_DIR"] / f"{save_name}.html"
        fig.write_html(save_path, include_plotlyjs="cdn")
        print(f"Saved: {save_path}")
    if SHOW_FIGURES:
        fig.show()
    return fig, summary


hvg_checkpoint_action0_by_seed, hvg_checkpoint_action0_missing = _collect_a0_hvg_best_test_checkpoint_action0_rows(
    A0_HVG_CHECKPOINT_VARIANTS
)
print(f"A0 HVG best-test checkpoint-aligned action-0 rows: {len(hvg_checkpoint_action0_by_seed)}")
if SHOW_DIAGNOSTIC_TABLES:
    display(hvg_checkpoint_action0_by_seed.sort_values(["order", "condition_label", "seed", "agent"]))
    if not hvg_checkpoint_action0_missing.empty:
        print("Missing / skipped A0 HVG checkpoint-aligned rows:")
        display(hvg_checkpoint_action0_missing)
elif not hvg_checkpoint_action0_missing.empty:
    print(f"A0 HVG missing/skipped rows hidden: {len(hvg_checkpoint_action0_missing)}. Set SHOW_DIAGNOSTIC_TABLES=True to inspect them.")

fig_a0_hvg_checkpoint_action0_heuristics, a0_hvg_checkpoint_action0_heuristics_summary = _plot_checkpoint_action0_grouped_bars(
    hvg_checkpoint_action0_by_seed,
    plot_group="heuristic",
    labels=[A0_HVG_BASELINE_LABEL, "global rho heuristic", "local rho heuristic"],
    title=f"A0 HVG: checkpoint-aligned action-0 fraction, {A0_HVG_BASELINE_LABEL} vs heuristic overrides",
    save_name="a0_hvg_checkpoint_action0_baseline_global_local_heuristic",
)

fig_a0_hvg_checkpoint_action0_gates, a0_hvg_checkpoint_action0_gates_summary = _plot_checkpoint_action0_grouped_bars(
    hvg_checkpoint_action0_by_seed,
    plot_group="gate",
    labels=[A0_HVG_BASELINE_LABEL, "gate final-action MAP", "gate hierarchical greedy"],
    title=f"A0 HVG: checkpoint-aligned action-0 fraction, {A0_HVG_BASELINE_LABEL} vs gate variants",
    save_name="a0_hvg_checkpoint_action0_baseline_gate_map_hierarchical",
)


A0 HVG best-test checkpoint-aligned action-0 rows: 45
Checkpoint files used for: A0 HVG: checkpoint-aligned action-0 fraction, a0_hvg_00_baseline vs heuristic overrides


,condition_label,run_name,seed,checkpoint_file,checkpoint_global_step,selected_metric_step,step_delta,step_match_policy,checkpoint_path,agent_0_plotted_action0_fraction,agent_1_plotted_action0_fraction,agent_2_plotted_action0_fraction,agent_0_mean_plotted_action0_fraction,agent_1_mean_plotted_action0_fraction,agent_2_mean_plotted_action0_fraction
0,a0_hvg_00_baseline,a0_hvg_00_baseline_s0,0,best_test_a0_hvg_00_baseline_s0.tar,13600000,13600000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.300781,0.145709,0.305370,0.555357,0.384102,0.368167
1,a0_hvg_00_baseline,a0_hvg_00_baseline_s1,1,best_test_a0_hvg_00_baseline_s1.tar,13040000,13040000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.450236,0.489993,0.469506,0.555357,0.384102,0.368167
2,a0_hvg_00_baseline,a0_hvg_00_baseline_s2,2,best_test_a0_hvg_00_baseline_s2.tar,13760000,13760000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.915055,0.516605,0.329625,0.555357,0.384102,0.368167
3,global rho heuristic,a0_hvg_01_eval_rho090_s0,0,best_test_a0_hvg_01_eval_rho090_s0.tar,13920000,13920000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.841109,0.843762,0.881337,0.930105,0.935384,0.937789
4,global rho heuristic,a0_hvg_01_eval_rho090_s1,1,best_test_a0_hvg_01_eval_rho090_s1.tar,14320000,14320000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.953398,0.995275,0.965079,0.930105,0.935384,0.937789
5,global rho heuristic,a0_hvg_01_eval_rho090_s2,2,best_test_a0_hvg_01_eval_rho090_s2.tar,14800000,14800000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.995809,0.967113,0.966952,0.930105,0.935384,0.937789
6,local rho heuristic,a0_hvg_04_eval_local_rho090_s0,0,best_test_a0_hvg_04_eval_local_rho090_s0.tar,14160000,14160000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.998500,0.960863,0.852245,0.992535,0.985656,0.926757
7,local rho heuristic,a0_hvg_04_eval_local_rho090_s1,1,best_test_a0_hvg_04_eval_local_rho090_s1.tar,14400000,14400000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.979241,0.999504,0.976141,0.992535,0.985656,0.926757
8,local rho heuristic,a0_hvg_04_eval_local_rho090_s2,2,best_test_a0_hvg_04_eval_local_rho090_s2.tar,14160000,14160000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.999864,0.996602,0.951885,0.992535,0.985656,0.926757


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/a0_hvg_checkpoint_action0_baseline_global_local_heuristic.html


Checkpoint files used for: A0 HVG: checkpoint-aligned action-0 fraction, a0_hvg_00_baseline vs gate variants


,condition_label,run_name,seed,checkpoint_file,checkpoint_global_step,selected_metric_step,step_delta,step_match_policy,checkpoint_path,agent_0_plotted_action0_fraction,agent_1_plotted_action0_fraction,agent_2_plotted_action0_fraction,agent_0_mean_plotted_action0_fraction,agent_1_mean_plotted_action0_fraction,agent_2_mean_plotted_action0_fraction
0,a0_hvg_00_baseline,a0_hvg_00_baseline_s0,0,best_test_a0_hvg_00_baseline_s0.tar,13600000,13600000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.300781,0.145709,0.305370,0.555357,0.384102,0.368167
1,a0_hvg_00_baseline,a0_hvg_00_baseline_s1,1,best_test_a0_hvg_00_baseline_s1.tar,13040000,13040000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.450236,0.489993,0.469506,0.555357,0.384102,0.368167
2,a0_hvg_00_baseline,a0_hvg_00_baseline_s2,2,best_test_a0_hvg_00_baseline_s2.tar,13760000,13760000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.915055,0.516605,0.329625,0.555357,0.384102,0.368167
3,gate final-action MAP,a0_hvg_02_gate_final_map_s0,0,best_test_a0_hvg_02_gate_final_map_s0.tar,12000000,12000000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.791306,0.858396,0.674725,0.822348,0.890514,0.686629
4,gate final-action MAP,a0_hvg_02_gate_final_map_s1,1,best_test_a0_hvg_02_gate_final_map_s1.tar,12720000,12720000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.878937,0.848861,0.668482,0.822348,0.890514,0.686629
5,gate final-action MAP,a0_hvg_02_gate_final_map_s2,2,best_test_a0_hvg_02_gate_final_map_s2.tar,13760000,13760000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.796801,0.964286,0.716679,0.822348,0.890514,0.686629
6,gate hierarchical greedy,a0_hvg_03_gate_hierarchical_s0,0,best_test_a0_hvg_03_gate_hierarchical_s0.tar,7120000,7120000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.092274,0.941543,0.185714,0.076157,0.843738,0.283428
7,gate hierarchical greedy,a0_hvg_03_gate_hierarchical_s1,1,best_test_a0_hvg_03_gate_hierarchical_s1.tar,14720000,14720000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.014571,0.629601,0.193304,0.076157,0.843738,0.283428
8,gate hierarchical greedy,a0_hvg_03_gate_hierarchical_s2,2,best_test_a0_hvg_03_gate_hierarchical_s2.tar,13760000,13760000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.121627,0.960069,0.471267,0.076157,0.843738,0.283428


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/a0_hvg_checkpoint_action0_baseline_gate_map_hierarchical.html


## A0 Sparse16 Checkpoint-Aligned Action-0 Plots

Same checkpoint-aligned action-0 / do-nothing plot style as above, but for the Sparse16 intervention-penalty runs. Bars show mean over seeds at each run's `best_test_<run>.tar` checkpoint; dots show the individual seed values. Flat and gated policies are separated into two plots.


In [23]:
A0_SPARSE16_CHECKPOINT_VARIANTS = [
    {
        "family": "a0_hvg_00_baseline",
        "label": A0_HVG_BASELINE_LABEL,
        "plot_group": "flat",
        "order": 0,
    },
    {
        "family": "a0_hvg_00_baseline",
        "label": A0_HVG_BASELINE_LABEL,
        "plot_group": "gated",
        "order": 0,
    },
    {
        "family": "a0_sparse16_flat_p001",
        "label": "flat p0.001",
        "plot_group": "flat",
        "order": 1,
    },
    {
        "family": "a0_sparse16_flat_p003",
        "label": "flat p0.003",
        "plot_group": "flat",
        "order": 2,
    },
    {
        "family": "a0_sparse16_flat_p010",
        "label": "flat p0.010",
        "plot_group": "flat",
        "order": 3,
    },
    {
        "family": "a0_sparse16_gated_p000",
        "label": "gated p0.000",
        "plot_group": "gated",
        "order": 0,
    },
    {
        "family": "a0_sparse16_gated_p001",
        "label": "gated p0.001",
        "plot_group": "gated",
        "order": 1,
    },
    {
        "family": "a0_sparse16_gated_p003",
        "label": "gated p0.003",
        "plot_group": "gated",
        "order": 2,
    },
    {
        "family": "a0_sparse16_gated_p010",
        "label": "gated p0.010",
        "plot_group": "gated",
        "order": 3,
    },
]

A0_SPARSE16_VARIANT_COLORS = {
    A0_HVG_BASELINE_LABEL: "#1f77b4",
    "flat p0.001": "#ff7f0e",
    "flat p0.003": "#2ca02c",
    "flat p0.010": "#d62728",
    "gated p0.000": "#8dc8f3",
    "gated p0.001": "#ff7f0e",
    "gated p0.003": "#2ca02c",
    "gated p0.010": "#d62728",
}

# The shared plotting helper above reads this color map by label.
A0_HVG_VARIANT_COLORS.update(A0_SPARSE16_VARIANT_COLORS)

# Sparse16 exists in both CPU and GPU run-data folders. Keep one source slice so
# identical run names are not counted twice. Unsplit folders, such as
# a0_hvg_00_baseline, are kept as shared comparison rows.
SPARSE16_CHECKPOINT_RUN_DATA_SOURCE = adm._normalize_run_data_source(RUN_DATA_SOURCE) or "gpu"
_sparse16_selected_runs_original = ctx["selected_runs"]
_sparse16_selected_runs = adm._filter_rows_by_run_data_source(
    _sparse16_selected_runs_original,
    SPARSE16_CHECKPOINT_RUN_DATA_SOURCE,
    keep_unsplit=True,
)
if _sparse16_selected_runs.empty:
    print(
        f"No Sparse16 rows found for source={SPARSE16_CHECKPOINT_RUN_DATA_SOURCE!r}; "
        "falling back to all loaded Sparse16 rows."
    )
    _sparse16_selected_runs = _sparse16_selected_runs_original
else:
    _sparse16_source_mask = _sparse16_selected_runs["run_name"].fillna("").astype(str).str.startswith("a0_sparse16_")
    _sparse16_baseline_source_mask = _sparse16_selected_runs["run_name"].fillna("").astype(str).str.startswith(f"{A0_HVG_BASELINE_LABEL}_")
    print(
        f"Sparse16 checkpoint plots using run-data source={SPARSE16_CHECKPOINT_RUN_DATA_SOURCE!r} "
        f"({_sparse16_source_mask.sum()} cached Sparse16 runs, "
        f"{_sparse16_selected_runs.loc[_sparse16_source_mask, 'run_name'].nunique()} unique Sparse16 run names; "
        f"{_sparse16_baseline_source_mask.sum()} cached {A0_HVG_BASELINE_LABEL} runs)."
    )

ctx["selected_runs"] = _sparse16_selected_runs
try:
    sparse16_checkpoint_action0_by_seed, sparse16_checkpoint_action0_missing = _collect_a0_hvg_best_test_checkpoint_action0_rows(
        A0_SPARSE16_CHECKPOINT_VARIANTS
    )
finally:
    ctx["selected_runs"] = _sparse16_selected_runs_original
print(f"A0 Sparse16 best-test checkpoint-aligned action-0 rows: {len(sparse16_checkpoint_action0_by_seed)}")
if SHOW_DIAGNOSTIC_TABLES:
    display(sparse16_checkpoint_action0_by_seed.sort_values(["plot_group", "order", "seed", "agent"]))
    if not sparse16_checkpoint_action0_missing.empty:
        print("Missing / skipped A0 Sparse16 checkpoint-aligned rows:")
        display(sparse16_checkpoint_action0_missing)
elif not sparse16_checkpoint_action0_missing.empty:
    print(f"A0 Sparse16 missing/skipped rows hidden: {len(sparse16_checkpoint_action0_missing)}. Set SHOW_DIAGNOSTIC_TABLES=True to inspect them.")

fig_a0_sparse16_checkpoint_action0_flat, a0_sparse16_checkpoint_action0_flat_summary = _plot_checkpoint_action0_grouped_bars(
    sparse16_checkpoint_action0_by_seed,
    plot_group="flat",
    labels=[A0_HVG_BASELINE_LABEL, "flat p0.001", "flat p0.003", "flat p0.010"],
    title="A0 Sparse16 flat: checkpoint-aligned action-0 fraction by intervention penalty",
    save_name="a0_sparse16_checkpoint_action0_flat_penalties",
)

fig_a0_sparse16_checkpoint_action0_gated, a0_sparse16_checkpoint_action0_gated_summary = _plot_checkpoint_action0_grouped_bars(
    sparse16_checkpoint_action0_by_seed,
    plot_group="gated",
    labels=[A0_HVG_BASELINE_LABEL, "gated p0.000", "gated p0.001", "gated p0.003", "gated p0.010"],
    title="A0 Sparse16 gated: checkpoint-aligned action-0 fraction by intervention penalty",
    save_name="a0_sparse16_checkpoint_action0_gated_penalties",
)

# display(fig_a0_sparse16_checkpoint_action0_flat)
# fig_a0_sparse16_checkpoint_action0_gated


Sparse16 checkpoint plots using run-data source='gpu' (24 cached Sparse16 runs, 24 unique Sparse16 run names; 3 cached a0_hvg_00_baseline runs).
A0 Sparse16 best-test checkpoint-aligned action-0 rows: 81
Checkpoint files used for: A0 Sparse16 flat: checkpoint-aligned action-0 fraction by intervention penalty


,condition_label,run_name,seed,checkpoint_file,checkpoint_global_step,selected_metric_step,step_delta,step_match_policy,checkpoint_path,agent_0_plotted_action0_fraction,agent_1_plotted_action0_fraction,agent_2_plotted_action0_fraction,agent_0_mean_plotted_action0_fraction,agent_1_mean_plotted_action0_fraction,agent_2_mean_plotted_action0_fraction
0,a0_hvg_00_baseline,a0_hvg_00_baseline_s0,0,best_test_a0_hvg_00_baseline_s0.tar,13600000,13600000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.300781,0.145709,0.305370,0.555357,0.384102,0.368167
1,a0_hvg_00_baseline,a0_hvg_00_baseline_s1,1,best_test_a0_hvg_00_baseline_s1.tar,13040000,13040000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.450236,0.489993,0.469506,0.555357,0.384102,0.368167
2,a0_hvg_00_baseline,a0_hvg_00_baseline_s2,2,best_test_a0_hvg_00_baseline_s2.tar,13760000,13760000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.915055,0.516605,0.329625,0.555357,0.384102,0.368167
3,flat p0.001,a0_sparse16_flat_p001_s0,0,best_test_a0_sparse16_flat_p001_s0.tar,10640000,10640000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.594010,0.489732,0.324306,0.642229,0.383581,0.440232
4,flat p0.001,a0_sparse16_flat_p001_s1,1,best_test_a0_sparse16_flat_p001_s1.tar,11520000,11520000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.998065,0.198611,0.598338,0.642229,0.383581,0.440232
5,flat p0.001,a0_sparse16_flat_p001_s2,2,best_test_a0_sparse16_flat_p001_s2.tar,11600000,11600000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.334611,0.462401,0.398053,0.642229,0.383581,0.440232
6,flat p0.003,a0_sparse16_flat_p003_s0,0,best_test_a0_sparse16_flat_p003_s0.tar,12000000,12000000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.327579,0.345275,0.727406,0.685710,0.356465,0.690571
7,flat p0.003,a0_sparse16_flat_p003_s1,1,best_test_a0_sparse16_flat_p003_s1.tar,12960000,12960000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.992584,0.257416,0.876525,0.685710,0.356465,0.690571
8,flat p0.003,a0_sparse16_flat_p003_s2,2,best_test_a0_sparse16_flat_p003_s2.tar,12480000,12480000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.736967,0.466704,0.467783,0.685710,0.356465,0.690571
9,flat p0.010,a0_sparse16_flat_p010_s0,0,best_test_a0_sparse16_flat_p010_s0.tar,8960000,8960000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.780295,0.942113,0.810379,0.908511,0.956242,0.865848


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/a0_sparse16_checkpoint_action0_flat_penalties.html


Checkpoint files used for: A0 Sparse16 gated: checkpoint-aligned action-0 fraction by intervention penalty


,condition_label,run_name,seed,checkpoint_file,checkpoint_global_step,selected_metric_step,step_delta,step_match_policy,checkpoint_path,agent_0_plotted_action0_fraction,agent_1_plotted_action0_fraction,agent_2_plotted_action0_fraction,agent_0_mean_plotted_action0_fraction,agent_1_mean_plotted_action0_fraction,agent_2_mean_plotted_action0_fraction
0,a0_hvg_00_baseline,a0_hvg_00_baseline_s0,0,best_test_a0_hvg_00_baseline_s0.tar,13600000,13600000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.300781,0.145709,0.305370,0.555357,0.384102,0.368167
1,a0_hvg_00_baseline,a0_hvg_00_baseline_s1,1,best_test_a0_hvg_00_baseline_s1.tar,13040000,13040000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.450236,0.489993,0.469506,0.555357,0.384102,0.368167
2,a0_hvg_00_baseline,a0_hvg_00_baseline_s2,2,best_test_a0_hvg_00_baseline_s2.tar,13760000,13760000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.915055,0.516605,0.329625,0.555357,0.384102,0.368167
3,gated p0.000,a0_sparse16_gated_p000_s0,0,best_test_a0_sparse16_gated_p000_s0.tar,8480000,8480000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.839903,0.939462,0.766252,0.801783,0.938846,0.647028
4,gated p0.000,a0_sparse16_gated_p000_s1,1,best_test_a0_sparse16_gated_p000_s1.tar,4320000,2320000.0,-2000000.0,at_or_before,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.943472,0.902835,0.712540,0.801783,0.938846,0.647028
5,gated p0.000,a0_sparse16_gated_p000_s2,2,best_test_a0_sparse16_gated_p000_s2.tar,6000000,6000000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.621972,0.974242,0.462292,0.801783,0.938846,0.647028
6,gated p0.001,a0_sparse16_gated_p001_s0,0,best_test_a0_sparse16_gated_p001_s0.tar,5760000,5760000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.940563,0.949068,0.795371,0.958001,0.837215,0.803393
7,gated p0.001,a0_sparse16_gated_p001_s1,1,best_test_a0_sparse16_gated_p001_s1.tar,7600000,7600000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.933620,0.927505,0.682828,0.958001,0.837215,0.803393
8,gated p0.001,a0_sparse16_gated_p001_s2,2,best_test_a0_sparse16_gated_p001_s2.tar,3280000,3280000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.999819,0.635072,0.931978,0.958001,0.837215,0.803393
9,gated p0.003,a0_sparse16_gated_p003_s0,0,best_test_a0_sparse16_gated_p003_s0.tar,8960000,8960000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.897839,0.912753,0.767187,0.932307,0.932041,0.802757


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/a0_sparse16_checkpoint_action0_gated_penalties.html


## A0 AIB Checkpoint-Aligned Action-0 Plots

Same checkpoint-aligned action-0 / do-nothing plot style as above, but for the Adaptive Intervention Budget runs. Bars show mean over seeds at each run's `best_test_<run>.tar` checkpoint; dots show the individual seed values. The first plot compares flat budget variants against the plain baseline; the second compares the gated AIB variant against the matched flat local target `0.20` and baseline.


In [24]:
A0_AIB_CHECKPOINT_VARIANTS = [
    {
        "family": "a0_hvg_00_baseline",
        "label": A0_HVG_BASELINE_LABEL,
        "plot_group": "flat",
        "order": 0,
    },
    {
        "family": "a0_aib_00_flat_local_t020",
        "label": "flat local t0.20",
        "plot_group": "flat",
        "order": 1,
    },
    {
        "family": "a0_aib_01_flat_local_t010",
        "label": "flat local t0.10",
        "plot_group": "flat",
        "order": 2,
    },
    {
        "family": "a0_aib_02_flat_local_t035",
        "label": "flat local t0.35",
        "plot_group": "flat",
        "order": 3,
    },
    {
        "family": "a0_aib_04_flat_nonidle_t020",
        "label": "flat non-idle t0.20",
        "plot_group": "flat",
        "order": 4,
    },
    {
        "family": "a0_aib_00_flat_local_t020",
        "label": "flat local t0.20",
        "plot_group": "gate",
        "order": 1,
    },
    {
        "family": "a0_aib_03_gate_hgreedy_sep_local_t020",
        "label": "gate h-greedy t0.20",
        "plot_group": "gate",
        "order": 2,
    },
]

A0_AIB_VARIANT_COLORS = {
    A0_HVG_BASELINE_LABEL: "#1f77b4",
    "flat local t0.20": "#ff7f0e",
    "flat local t0.10": "#2ca02c",
    "flat local t0.35": "#d62728",
    "flat non-idle t0.20": "#9467bd",
    "gate h-greedy t0.20": "#8c564b",
}

# The shared plotting helper above reads this color map by label.
A0_HVG_VARIANT_COLORS.update(A0_AIB_VARIANT_COLORS)

# AIB exists in both CPU and GPU run-data folders. Keep one source slice so
# identical run names are not counted twice. Unsplit folders, such as the plain
# A0 HVG baseline, are kept as shared baselines.
AIB_CHECKPOINT_RUN_DATA_SOURCE = adm._normalize_run_data_source(RUN_DATA_SOURCE) or "gpu"
_aib_selected_runs_original = ctx["selected_runs"]
_aib_selected_runs = adm._filter_rows_by_run_data_source(
    _aib_selected_runs_original,
    AIB_CHECKPOINT_RUN_DATA_SOURCE,
    keep_unsplit=True,
)
if _aib_selected_runs.empty:
    print(
        f"No AIB rows found for source={AIB_CHECKPOINT_RUN_DATA_SOURCE!r}; "
        "falling back to all loaded rows."
    )
    _aib_selected_runs = _aib_selected_runs_original
else:
    _aib_source_mask = _aib_selected_runs["run_name"].fillna("").astype(str).str.startswith("a0_aib_")
    _baseline_source_mask = _aib_selected_runs["run_name"].fillna("").astype(str).str.startswith("a0_hvg_00_baseline_")
    print(
        f"AIB checkpoint plots using run-data source={AIB_CHECKPOINT_RUN_DATA_SOURCE!r} "
        f"({_aib_source_mask.sum()} cached AIB runs, "
        f"{_aib_selected_runs.loc[_aib_source_mask, 'run_name'].nunique()} unique AIB run names; "
        f"{_baseline_source_mask.sum()} cached plain-baseline runs)."
    )

ctx["selected_runs"] = _aib_selected_runs
try:
    aib_checkpoint_action0_by_seed, aib_checkpoint_action0_missing = _collect_a0_hvg_best_test_checkpoint_action0_rows(
        A0_AIB_CHECKPOINT_VARIANTS
    )
finally:
    ctx["selected_runs"] = _aib_selected_runs_original
print(f"A0 AIB best-test checkpoint-aligned action-0 rows: {len(aib_checkpoint_action0_by_seed)}")
if SHOW_DIAGNOSTIC_TABLES:
    display(aib_checkpoint_action0_by_seed.sort_values(["plot_group", "order", "seed", "agent"]))
    if not aib_checkpoint_action0_missing.empty:
        print("Missing / skipped A0 AIB checkpoint-aligned rows:")
        display(aib_checkpoint_action0_missing)
elif not aib_checkpoint_action0_missing.empty:
    print(f"A0 AIB missing/skipped rows hidden: {len(aib_checkpoint_action0_missing)}. Set SHOW_DIAGNOSTIC_TABLES=True to inspect them.")

fig_a0_aib_checkpoint_action0_flat, a0_aib_checkpoint_action0_flat_summary = _plot_checkpoint_action0_grouped_bars(
    aib_checkpoint_action0_by_seed,
    plot_group="flat",
    labels=[A0_HVG_BASELINE_LABEL, "flat local t0.10", "flat local t0.20", "flat local t0.35", "flat non-idle t0.20"],
    title="A0 AIB flat budget variants: checkpoint-aligned action-0 fraction",
    save_name="a0_aib_checkpoint_action0_flat_budget_variants",
)

fig_a0_aib_checkpoint_action0_gate, a0_aib_checkpoint_action0_gate_summary = _plot_checkpoint_action0_grouped_bars(
    aib_checkpoint_action0_by_seed,
    plot_group="gate",
    labels=[A0_HVG_BASELINE_LABEL, "flat local t0.20", "gate h-greedy t0.20"],
    title="A0 AIB gated budget variant: checkpoint-aligned action-0 fraction",
    save_name="a0_aib_checkpoint_action0_gate_budget_variant",
)

# display(fig_a0_aib_checkpoint_action0_flat)
# fig_a0_aib_checkpoint_action0_gate


AIB checkpoint plots using run-data source='gpu' (15 cached AIB runs, 15 unique AIB run names; 3 cached plain-baseline runs).
A0 AIB best-test checkpoint-aligned action-0 rows: 63
Checkpoint files used for: A0 AIB flat budget variants: checkpoint-aligned action-0 fraction


,condition_label,run_name,seed,checkpoint_file,checkpoint_global_step,selected_metric_step,step_delta,step_match_policy,checkpoint_path,agent_0_plotted_action0_fraction,agent_1_plotted_action0_fraction,agent_2_plotted_action0_fraction,agent_0_mean_plotted_action0_fraction,agent_1_mean_plotted_action0_fraction,agent_2_mean_plotted_action0_fraction
0,a0_hvg_00_baseline,a0_hvg_00_baseline_s0,0,best_test_a0_hvg_00_baseline_s0.tar,13600000,13600000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.300781,0.145709,0.305370,0.555357,0.384102,0.368167
1,a0_hvg_00_baseline,a0_hvg_00_baseline_s1,1,best_test_a0_hvg_00_baseline_s1.tar,13040000,13040000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.450236,0.489993,0.469506,0.555357,0.384102,0.368167
2,a0_hvg_00_baseline,a0_hvg_00_baseline_s2,2,best_test_a0_hvg_00_baseline_s2.tar,13760000,13760000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.915055,0.516605,0.329625,0.555357,0.384102,0.368167
3,flat local t0.10,a0_aib_01_flat_local_t010_s0,0,best_test_a0_aib_01_flat_local_t010_s0.tar,12880000,12880000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.997470,0.999182,0.948016,0.990728,0.994382,0.946813
4,flat local t0.10,a0_aib_01_flat_local_t010_s1,1,best_test_a0_aib_01_flat_local_t010_s1.tar,13200000,13200000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.999876,0.994296,0.918155,0.990728,0.994382,0.946813
5,flat local t0.10,a0_aib_01_flat_local_t010_s2,2,best_test_a0_aib_01_flat_local_t010_s2.tar,12720000,12720000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.974839,0.989670,0.974268,0.990728,0.994382,0.946813
6,flat local t0.20,a0_aib_00_flat_local_t020_s0,0,best_test_a0_aib_00_flat_local_t020_s0.tar,13120000,13120000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.986136,0.991233,0.954675,0.982048,0.989906,0.966997
7,flat local t0.20,a0_aib_00_flat_local_t020_s1,1,best_test_a0_aib_00_flat_local_t020_s1.tar,9360000,9360000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.988852,0.984611,0.964633,0.982048,0.989906,0.966997
8,flat local t0.20,a0_aib_00_flat_local_t020_s2,2,best_test_a0_aib_00_flat_local_t020_s2.tar,12160000,12160000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.971156,0.993874,0.981684,0.982048,0.989906,0.966997
9,flat local t0.35,a0_aib_02_flat_local_t035_s0,0,best_test_a0_aib_02_flat_local_t035_s0.tar,12800000,12800000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.955965,0.900893,0.924392,0.957188,0.949314,0.895718


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/a0_aib_checkpoint_action0_flat_budget_variants.html


Checkpoint files used for: A0 AIB gated budget variant: checkpoint-aligned action-0 fraction


,condition_label,run_name,seed,checkpoint_file,checkpoint_global_step,selected_metric_step,step_delta,step_match_policy,checkpoint_path,agent_0_plotted_action0_fraction,agent_1_plotted_action0_fraction,agent_2_plotted_action0_fraction,agent_0_mean_plotted_action0_fraction,agent_1_mean_plotted_action0_fraction,agent_2_mean_plotted_action0_fraction
0,a0_hvg_00_baseline,a0_hvg_00_baseline_s0,0,best_test_a0_hvg_00_baseline_s0.tar,13600000,13600000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.300781,0.145709,0.305370,0.555357,0.384102,0.368167
1,a0_hvg_00_baseline,a0_hvg_00_baseline_s1,1,best_test_a0_hvg_00_baseline_s1.tar,13040000,13040000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.450236,0.489993,0.469506,0.555357,0.384102,0.368167
2,a0_hvg_00_baseline,a0_hvg_00_baseline_s2,2,best_test_a0_hvg_00_baseline_s2.tar,13760000,13760000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.915055,0.516605,0.329625,0.555357,0.384102,0.368167
3,flat local t0.20,a0_aib_00_flat_local_t020_s0,0,best_test_a0_aib_00_flat_local_t020_s0.tar,13120000,13120000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.986136,0.991233,0.954675,0.982048,0.989906,0.966997
4,flat local t0.20,a0_aib_00_flat_local_t020_s1,1,best_test_a0_aib_00_flat_local_t020_s1.tar,9360000,9360000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.988852,0.984611,0.964633,0.982048,0.989906,0.966997
5,flat local t0.20,a0_aib_00_flat_local_t020_s2,2,best_test_a0_aib_00_flat_local_t020_s2.tar,12160000,12160000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.971156,0.993874,0.981684,0.982048,0.989906,0.966997
6,gate h-greedy t0.20,a0_aib_03_gate_hgreedy_sep_local_t020_s0,0,best_test_a0_aib_03_gate_hgreedy_sep_local_t02...,14960000,14960000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.999935,0.999787,0.996891,0.999873,0.999929,0.998538
7,gate h-greedy t0.20,a0_aib_03_gate_hgreedy_sep_local_t020_s1,1,best_test_a0_aib_03_gate_hgreedy_sep_local_t02...,5280000,5280000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,1.000000,1.000000,0.999451,0.999873,0.999929,0.998538
8,gate h-greedy t0.20,a0_aib_03_gate_hgreedy_sep_local_t020_s2,2,best_test_a0_aib_03_gate_hgreedy_sep_local_t02...,16800000,16800000.0,0.0,exact,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.999684,1.000000,0.999273,0.999873,0.999929,0.998538


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/a0_aib_checkpoint_action0_gate_budget_variant.html


## Teacher-Student BC Action-0 Usage by Agent

Teacher-student checkpoints are offline BC checkpoints, not normal W&B training runs, so they do not have the same rollout action-distribution history as MAPPO runs. This plot compares the checkpoint-aligned `a0_hvg_00_baseline`, the teacher targets from `teacher_student_epoch_metrics.csv` (`1 - teacher_nonidle_frac`), and the final BC eval predictions (`1 - pred_nonidle_frac`). Bars show the mean over seeds; dots show the actual seed values.


In [25]:
import warnings

TEACHER_STUDENT_EPOCH_METRICS_PATH = TASK_DIR / "outputs" / "comparison_dashboard_figures" / "teacher_student_epoch_metrics.csv"

TEACHER_STUDENT_TEACHER_LABEL = "teacher"

TEACHER_STUDENT_FAMILY_LABELS = {
    "local_bc": "BC bal0.50 w5 aux0.50",
    "local_bc_bal020_w3_aux025": "BC bal0.20 w3 aux0.25",
}
TEACHER_STUDENT_FAMILY_ORDER = {
    "local_bc": 2,
    "local_bc_bal020_w3_aux025": 3,
}
TEACHER_STUDENT_COLORS = {
    TEACHER_STUDENT_TEACHER_LABEL: "#2ca02c",
    "BC bal0.50 w5 aux0.50": "#ff7f0e",
    "BC bal0.20 w3 aux0.25": "#9467bd",
}
A0_HVG_VARIANT_COLORS.update(TEACHER_STUDENT_COLORS)

if not TEACHER_STUDENT_EPOCH_METRICS_PATH.exists():
    raise FileNotFoundError(
        f"Teacher-student epoch metrics not found: {TEACHER_STUDENT_EPOCH_METRICS_PATH}"
    )

teacher_student_epoch_metrics = pd.read_csv(TEACHER_STUDENT_EPOCH_METRICS_PATH)
required_columns = {
    "checkpoint",
    "checkpoint_family",
    "checkpoint_seed",
    "phase",
    "epoch",
    "agent",
    "dataset",
    "pred_nonidle_frac",
    "teacher_nonidle_frac",
}
missing_columns = required_columns.difference(teacher_student_epoch_metrics.columns)
if missing_columns:
    raise KeyError(f"Missing teacher-student epoch metric columns: {sorted(missing_columns)}")

teacher_student_eval = teacher_student_epoch_metrics[
    teacher_student_epoch_metrics["phase"].astype(str).eq("eval")
].copy()
teacher_student_eval["epoch"] = pd.to_numeric(teacher_student_eval["epoch"], errors="coerce")
teacher_student_eval["pred_nonidle_frac"] = pd.to_numeric(
    teacher_student_eval["pred_nonidle_frac"], errors="coerce"
)
teacher_student_eval = teacher_student_eval.dropna(
    subset=["epoch", "pred_nonidle_frac", "checkpoint_family", "checkpoint_seed", "agent"]
)

if teacher_student_eval.empty:
    raise RuntimeError("No teacher-student eval rows with pred_nonidle_frac were found.")

_final_epoch_by_checkpoint = teacher_student_eval.groupby("checkpoint", dropna=False)["epoch"].transform("max")
teacher_student_final = teacher_student_eval[teacher_student_eval["epoch"].eq(_final_epoch_by_checkpoint)].copy()
teacher_student_final["condition_label"] = teacher_student_final["checkpoint_family"].map(TEACHER_STUDENT_FAMILY_LABELS).fillna(
    teacher_student_final["checkpoint_family"].astype(str)
)
teacher_student_final["order"] = teacher_student_final["checkpoint_family"].map(TEACHER_STUDENT_FAMILY_ORDER).fillna(99).astype(int)
teacher_student_final["plot_group"] = "teacher_student"
teacher_student_final["seed"] = pd.to_numeric(teacher_student_final["checkpoint_seed"], errors="coerce").astype("Int64")
teacher_student_final["action0_fraction"] = 1.0 - teacher_student_final["pred_nonidle_frac"]
teacher_student_final["run_name"] = teacher_student_final["checkpoint"].astype(str).str.replace(r"\.tar$", "", regex=True)
teacher_student_final["run_id"] = teacher_student_final["run_name"]
_teacher_student_optimizer_steps = teacher_student_final.get(
    "optimizer_steps",
    pd.Series([pd.NA] * len(teacher_student_final), index=teacher_student_final.index),
)
teacher_student_final["checkpoint_global_step"] = _teacher_student_optimizer_steps.values
teacher_student_final["selected_metric_step"] = _teacher_student_optimizer_steps.values
teacher_student_final["step_delta"] = 0
teacher_student_final["metric_column"] = "teacher_student_epoch_metrics.pred_nonidle_frac"
teacher_student_final["metric_inverted"] = True


def _teacher_student_checkpoint_path(checkpoint_name):
    candidate = TASK_DIR / "checkpoint" / "teacher_student" / str(checkpoint_name)
    return str(candidate if candidate.exists() else checkpoint_name)


teacher_student_final["checkpoint_path"] = teacher_student_final["checkpoint"].map(_teacher_student_checkpoint_path)

teacher_student_action0_by_seed = teacher_student_final[[
    "run_name",
    "run_id",
    "checkpoint",
    "checkpoint_family",
    "condition_label",
    "plot_group",
    "order",
    "seed",
    "agent",
    "action0_fraction",
    "pred_nonidle_frac",
    "teacher_nonidle_frac",
    "accuracy",
    "false_noop_rate",
    "false_intervention_rate",
    "checkpoint_global_step",
    "selected_metric_step",
    "step_delta",
    "metric_column",
    "metric_inverted",
    "checkpoint_path",
]].copy()

teacher_action0_by_seed = teacher_student_final.dropna(subset=["teacher_nonidle_frac"]).copy()
teacher_action0_by_seed = (
    teacher_action0_by_seed
    .sort_values(["seed", "agent", "checkpoint_family"], kind="stable")
    .drop_duplicates(["seed", "agent"])
)
teacher_action0_by_seed["condition_label"] = TEACHER_STUDENT_TEACHER_LABEL
teacher_action0_by_seed["plot_group"] = "teacher_student"
teacher_action0_by_seed["order"] = 1
teacher_action0_by_seed["action0_fraction"] = 1.0 - pd.to_numeric(teacher_action0_by_seed["teacher_nonidle_frac"], errors="coerce")
teacher_action0_by_seed["run_name"] = "teacher_" + teacher_action0_by_seed["dataset"].astype(str)
teacher_action0_by_seed["run_id"] = teacher_action0_by_seed["run_name"]
teacher_action0_by_seed["checkpoint"] = teacher_action0_by_seed["dataset"].astype(str)
teacher_action0_by_seed["checkpoint_path"] = teacher_action0_by_seed["dataset"].astype(str)
teacher_action0_by_seed["checkpoint_global_step"] = pd.NA
teacher_action0_by_seed["selected_metric_step"] = pd.NA
teacher_action0_by_seed["step_delta"] = 0
teacher_action0_by_seed["metric_column"] = "teacher_student_epoch_metrics.teacher_nonidle_frac"
teacher_action0_by_seed["metric_inverted"] = True
teacher_action0_by_seed["pred_nonidle_frac"] = pd.NA
teacher_action0_by_seed["accuracy"] = pd.NA
teacher_action0_by_seed["false_noop_rate"] = pd.NA
teacher_action0_by_seed["false_intervention_rate"] = pd.NA
teacher_action0_by_seed = teacher_action0_by_seed[teacher_student_action0_by_seed.columns].copy()

baseline_teacher_student_action0 = baseline_action0_by_seed.copy()
baseline_teacher_student_action0["condition_label"] = A0_HVG_BASELINE_LABEL
baseline_teacher_student_action0["plot_group"] = "teacher_student"
baseline_teacher_student_action0["order"] = 0
baseline_teacher_student_action0["checkpoint"] = baseline_teacher_student_action0["checkpoint_path"].map(lambda value: Path(value).name if pd.notna(value) and str(value) else "")
baseline_teacher_student_action0["checkpoint_family"] = A0_HVG_BASELINE_LABEL
baseline_teacher_student_action0["pred_nonidle_frac"] = pd.NA
baseline_teacher_student_action0["teacher_nonidle_frac"] = pd.NA
baseline_teacher_student_action0["accuracy"] = pd.NA
baseline_teacher_student_action0["false_noop_rate"] = pd.NA
baseline_teacher_student_action0["false_intervention_rate"] = pd.NA
baseline_teacher_student_action0["metric_inverted"] = baseline_teacher_student_action0.get("metric_inverted_from_nonidle", False)
baseline_teacher_student_action0 = baseline_teacher_student_action0[teacher_student_action0_by_seed.columns].copy()

with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated",
        category=FutureWarning,
    )
    teacher_student_action0_by_seed = pd.concat(
        [baseline_teacher_student_action0, teacher_action0_by_seed, teacher_student_action0_by_seed],
        ignore_index=True,
        sort=False,
    )

print(f"Teacher-student action-0 comparison rows: {len(teacher_student_action0_by_seed)}")
if SHOW_DIAGNOSTIC_TABLES:
    display(teacher_student_action0_by_seed.sort_values(["order", "seed", "agent"]))

fig_teacher_student_action0_by_agent, teacher_student_action0_summary = _plot_checkpoint_action0_grouped_bars(
    teacher_student_action0_by_seed,
    plot_group="teacher_student",
    labels=[A0_HVG_BASELINE_LABEL, TEACHER_STUDENT_TEACHER_LABEL, "BC bal0.50 w5 aux0.50", "BC bal0.20 w3 aux0.25"],
    title="Teacher-student BC: action-0 fraction by agent vs baseline and teacher",
    save_name="teacher_student_bc_final_eval_action0_by_agent",
)

# fig_teacher_student_action0_by_agent


Teacher-student action-0 comparison rows: 36
Checkpoint files used for: Teacher-student BC: action-0 fraction by agent vs baseline and teacher


,condition_label,run_name,seed,checkpoint_file,checkpoint_global_step,selected_metric_step,step_delta,step_match_policy,checkpoint_path,agent_0_plotted_action0_fraction,agent_1_plotted_action0_fraction,agent_2_plotted_action0_fraction,agent_0_mean_plotted_action0_fraction,agent_1_mean_plotted_action0_fraction,agent_2_mean_plotted_action0_fraction
0,a0_hvg_00_baseline,a0_hvg_00_baseline_s0,0,best_test_a0_hvg_00_baseline_s0.tar,13600000,13600000.0,0.0,,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.300781,0.145709,0.305370,0.555357,0.384102,0.368167
1,a0_hvg_00_baseline,a0_hvg_00_baseline_s1,1,best_test_a0_hvg_00_baseline_s1.tar,13040000,13040000.0,0.0,,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.450236,0.489993,0.469506,0.555357,0.384102,0.368167
2,a0_hvg_00_baseline,a0_hvg_00_baseline_s2,2,best_test_a0_hvg_00_baseline_s2.tar,13760000,13760000.0,0.0,,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.915055,0.516605,0.329625,0.555357,0.384102,0.368167
3,teacher,teacher_local_rho090_s0,0,local_rho090_s0,<NA>,NaN,0.0,,local_rho090_s0,NaN,NaN,NaN,0.990495,0.961864,0.950489
4,teacher,teacher_local_rho090_s1,1,local_rho090_s1,<NA>,NaN,0.0,,local_rho090_s1,NaN,NaN,NaN,0.990495,0.961864,0.950489
5,teacher,teacher_local_rho090_s2,2,local_rho090_s2,<NA>,NaN,0.0,,local_rho090_s2,NaN,NaN,NaN,0.990495,0.961864,0.950489
6,BC bal0.50 w5 aux0.50,local_bc_s0,0,local_bc_s0.tar,12375,12375.0,0.0,,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.944898,0.853673,0.854267,0.969392,0.926127,0.911873
7,BC bal0.50 w5 aux0.50,local_bc_s1,1,local_bc_s1.tar,14994,14994.0,0.0,,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.966563,0.985901,0.945575,0.969392,0.926127,0.911873
8,BC bal0.50 w5 aux0.50,local_bc_s2,2,local_bc_s2.tar,13617,13617.0,0.0,,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.996714,0.938807,0.935777,0.969392,0.926127,0.911873
9,BC bal0.20 w3 aux0.25,local_bc_bal020_w3_aux025_s0,0,local_bc_bal020_w3_aux025_s0.tar,12375,12375.0,0.0,,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.968994,0.878430,0.882885,0.979279,0.944206,0.934988


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/teacher_student_bc_final_eval_action0_by_agent.html


## Checkpoint-Aligned Action-0 / Survival Tradeoff

These plots reproduce the Action-0 / Survival Tradeoff idea from the CPU action-distribution notebook, but they use the checkpoint-aligned `best_test_<run>.tar` rows built above. Each seed point is the mean action-0 fraction across agents at that checkpoint versus the episodic survival metric selected at the same checkpoint step. Diamonds show the condition mean over seeds.


In [26]:
SURVIVAL_METRIC_CANDIDATES = [
    "test/charts/episodic_survival",
    "test/episodic_survival",
    "validation/episodic_survival",
    "charts/episodic_survival",
    "train_eval/charts/episodic_survival",
    "train_eval/episodic_survival",
]


def _checkpoint_tradeoff_survival_scale(values):
    numeric = pd.to_numeric(values, errors="coerce")
    max_value = numeric.max(skipna=True)
    if pd.isna(max_value):
        return 100.0
    return 100.0 if max_value <= 1.5 else 1.0


def _plot_range_with_padding(values, *, lower=None, upper=None, pad_fraction=0.12, min_span=0.05):
    numeric = pd.to_numeric(pd.Series(values), errors="coerce").dropna()
    if numeric.empty:
        return [lower, upper] if lower is not None and upper is not None else None

    vmin = float(numeric.min())
    vmax = float(numeric.max())
    span = max(vmax - vmin, float(min_span))
    pad = span * float(pad_fraction)
    lo = vmin - pad
    hi = vmax + pad

    if lower is not None:
        lo = max(float(lower), lo)
    if upper is not None:
        hi = min(float(upper), hi)
    if hi <= lo:
        hi = lo + float(min_span)
        if upper is not None and hi > float(upper):
            hi = float(upper)
            lo = max(float(lower) if lower is not None else hi - float(min_span), hi - float(min_span))
    return [lo, hi]


def _checkpoint_tradeoff_filter_rows(seed_rows, *, labels=None, plot_group=None):
    if seed_rows is None or seed_rows.empty:
        return pd.DataFrame()
    data = seed_rows.copy()
    if labels is not None and "condition_label" in data.columns:
        data = data[data["condition_label"].astype(str).isin([str(label) for label in labels])].copy()
    if plot_group is not None and "plot_group" in data.columns:
        baseline_mask = data.get("condition_label", pd.Series(index=data.index, dtype=object)).astype(str).eq(A0_HVG_BASELINE_LABEL)
        data = data[data["plot_group"].astype(str).eq(str(plot_group)) | baseline_mask].copy()
        data["plot_group"] = str(plot_group)
    return data


def _checkpoint_survival_point_for_run(run_history, checkpoint_step):
    if run_history is None or run_history.empty:
        return None
    for metric in SURVIVAL_METRIC_CANDIDATES:
        if metric not in run_history.columns:
            continue
        values = pd.to_numeric(run_history[metric], errors="coerce")
        if not values.notna().any():
            continue
        selected = adm._select_checkpoint_metric_point(
            run_history,
            values,
            checkpoint_step,
            step_policy=STEP_MATCH_POLICY,
        )
        if selected is None:
            continue
        point, match_policy = selected
        scale = _checkpoint_tradeoff_survival_scale(values)
        selected_step = float(point["_step"])
        return {
            "survival_pct": float(point["value"]) * scale,
            "survival_metric": metric,
            "survival_selected_metric_step": selected_step,
            "survival_selected_metric_step_millions": selected_step / 1_000_000,
            "survival_step_delta": selected_step - float(checkpoint_step),
            "survival_step_match_policy": match_policy,
        }
    return None


def _checkpoint_action0_survival_tradeoff_rows(seed_rows, *, labels=None, plot_group=None):
    plot_rows = _checkpoint_tradeoff_filter_rows(seed_rows, labels=labels, plot_group=plot_group)
    if plot_rows.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    plot_rows["action0_fraction"] = pd.to_numeric(plot_rows["action0_fraction"], errors="coerce")
    group_cols = [
        "plot_group",
        "condition_label",
        "order",
        "run_name",
        "run_id",
        "seed",
        "checkpoint_path",
        "checkpoint_global_step",
        "selected_metric_step",
        "step_delta",
        "step_match_policy",
    ]
    group_cols = [col for col in group_cols if col in plot_rows.columns]
    per_run = (
        plot_rows
        .dropna(subset=["action0_fraction"])
        .groupby(group_cols, dropna=False, as_index=False)
        .agg(
            mean_action0_fraction=("action0_fraction", "mean"),
            std_agent_action0_fraction=("action0_fraction", "std"),
            n_agents=("agent", "nunique"),
            agents=("agent", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
        )
    )
    if per_run.empty:
        return per_run, pd.DataFrame(), pd.DataFrame()
    per_run["std_agent_action0_fraction"] = per_run["std_agent_action0_fraction"].fillna(0.0)
    per_run["checkpoint_file"] = per_run["checkpoint_path"].fillna("").astype(str).map(lambda value: Path(value).name if value else "")

    history = ctx["history_wide"].copy()
    survival_rows = []
    missing_rows = []
    for row in per_run.to_dict("records"):
        run_id = str(row.get("run_id", ""))
        checkpoint_step = row.get("checkpoint_global_step")
        run_history = history[history["run_id"].astype(str).eq(run_id)].sort_values("_step").copy()
        survival = _checkpoint_survival_point_for_run(run_history, checkpoint_step)
        if survival is None:
            missing_rows.append({
                "condition_label": row.get("condition_label"),
                "run_name": row.get("run_name"),
                "run_id": row.get("run_id"),
                "seed": row.get("seed"),
                "checkpoint_path": row.get("checkpoint_path"),
                "checkpoint_global_step": row.get("checkpoint_global_step"),
                "reason": "missing_checkpoint_survival_metric",
            })
            continue
        survival_rows.append({**row, **survival})

    tradeoff = pd.DataFrame(survival_rows)
    missing = pd.DataFrame(missing_rows)
    if tradeoff.empty:
        return tradeoff, pd.DataFrame(), missing

    agg = (
        tradeoff
        .groupby(["plot_group", "condition_label", "order"], dropna=False, as_index=False)
        .agg(
            mean_action0_fraction=("mean_action0_fraction", "mean"),
            std_action0_fraction=("mean_action0_fraction", "std"),
            mean_survival_pct=("survival_pct", "mean"),
            std_survival_pct=("survival_pct", "std"),
            n_seeds=("run_id", "nunique"),
            seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
            runs=("run_name", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
            checkpoint_files=("checkpoint_file", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
            survival_metrics=("survival_metric", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
        )
        .sort_values(["plot_group", "order", "condition_label"])
    )
    agg["std_action0_fraction"] = agg["std_action0_fraction"].fillna(0.0)
    agg["std_survival_pct"] = agg["std_survival_pct"].fillna(0.0)
    return tradeoff, agg, missing


def _display_checkpoint_tradeoff_audit(tradeoff, agg, *, title):
    if tradeoff.empty:
        return pd.DataFrame()
    mean_cols = agg[[
        "plot_group",
        "condition_label",
        "mean_action0_fraction",
        "mean_survival_pct",
        "n_seeds",
        "seeds",
    ]].rename(columns={
        "mean_action0_fraction": "condition_mean_plotted_action0_fraction",
        "mean_survival_pct": "condition_mean_plotted_survival_pct",
    })
    audit = tradeoff.merge(mean_cols, on=["plot_group", "condition_label"], how="left")
    audit = audit.rename(columns={
        "mean_action0_fraction": "seed_plotted_action0_fraction",
        "survival_pct": "seed_plotted_survival_pct",
    })
    keep = [
        "condition_label",
        "run_name",
        "seed",
        "checkpoint_file",
        "checkpoint_global_step",
        "checkpoint_path",
        "seed_plotted_action0_fraction",
        "seed_plotted_survival_pct",
        "condition_mean_plotted_action0_fraction",
        "condition_mean_plotted_survival_pct",
        "n_seeds",
        "seeds",
        "survival_metric",
        "selected_metric_step",
        "survival_selected_metric_step",
        "step_delta",
        "survival_step_delta",
    ]
    keep = [col for col in keep if col in audit.columns]
    audit = audit[keep].sort_values(["condition_label", "seed", "run_name"], kind="stable").reset_index(drop=True)
    if SHOW_CHECKPOINT_AUDIT_TABLES:
        print(f"Checkpoint tradeoff values used for: {title}")
        display(audit)
    return audit


def _plot_checkpoint_action0_survival_tradeoff(seed_rows, *, plot_group, labels, title, save_name):
    tradeoff, agg, missing = _checkpoint_action0_survival_tradeoff_rows(
        seed_rows,
        labels=labels,
        plot_group=plot_group,
    )
    if not missing.empty:
        print(f"{title}: missing survival for {len(missing)} checkpoint/run rows.")
        if SHOW_DIAGNOSTIC_TABLES:
            display(missing)
    if tradeoff.empty or agg.empty:
        print(f"No checkpoint-aligned survival/action-0 tradeoff data available for: {title}")
        return None, tradeoff, agg, missing

    audit = _display_checkpoint_tradeoff_audit(tradeoff, agg, title=title)
    label_order = [label for label in labels if label in set(agg["condition_label"].astype(str))]
    palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
    colors = {
        label: A0_HVG_VARIANT_COLORS.get(label, palette[idx % len(palette)])
        for idx, label in enumerate(label_order)
    }

    fig = go.Figure()
    for label in label_order:
        color = colors[label]
        seed_points = tradeoff[tradeoff["condition_label"].astype(str).eq(label)].sort_values("seed")
        if not seed_points.empty:
            fig.add_trace(
                go.Scatter(
                    x=seed_points["mean_action0_fraction"],
                    y=seed_points["survival_pct"],
                    mode="markers",
                    name=f"{label} seeds",
                    legendgroup=label,
                    showlegend=False,
                    marker={
                        "color": color,
                        "size": 5.5,
                        "opacity": 0.48,
                        "line": {"color": "white", "width": 0.9},
                    },
                    customdata=np.stack([
                        seed_points["run_name"].astype(str),
                        seed_points["seed"],
                        seed_points["checkpoint_file"].astype(str),
                        seed_points["checkpoint_global_step"],
                        seed_points["survival_metric"].astype(str),
                        seed_points["agents"].astype(str),
                    ], axis=-1),
                    hovertemplate=(
                        "run=%{customdata[0]}<br>"
                        "seed=%{customdata[1]}<br>"
                        "checkpoint=%{customdata[2]}<br>"
                        "checkpoint_step=%{customdata[3]}<br>"
                        "mean action-0=%{x:.4f}<br>"
                        "survival=%{y:.2f}%<br>"
                        "survival metric=%{customdata[4]}<br>"
                        "agents=%{customdata[5]}<extra></extra>"
                    ),
                )
            )

        mean_row = agg[agg["condition_label"].astype(str).eq(label)].iloc[0]
        fig.add_trace(
            go.Scatter(
                x=[mean_row["mean_action0_fraction"]],
                y=[mean_row["mean_survival_pct"]],
                mode="markers",
                name=label,
                legendgroup=label,
                showlegend=True,
                marker={
                    "color": color,
                    "size": 12,
                    "symbol": "diamond",
                    "line": {"color": "black", "width": 1.1},
                },
                error_x={"type": "data", "array": [float(mean_row["std_action0_fraction"])]},
                error_y={"type": "data", "array": [float(mean_row["std_survival_pct"])]},
                customdata=np.array([[
                    mean_row["n_seeds"],
                    str(mean_row["seeds"]),
                    str(mean_row["runs"]),
                    str(mean_row["survival_metrics"]),
                ]], dtype=object),
                hovertemplate=(
                    f"<b>{label}</b><br>"
                    "mean action-0=%{x:.4f}<br>"
                    "mean survival=%{y:.2f}%<br>"
                    "n_seeds=%{customdata[0]}<br>"
                    "seeds=%{customdata[1]}<br>"
                    "runs=%{customdata[2]}<br>"
                    "survival metrics=%{customdata[3]}<extra></extra>"
                ),
            )
        )

    fig.add_annotation(
        text="better: higher survival and more action 0",
        x=0.98,
        y=102,
        xref="x",
        yref="y",
        showarrow=True,
        ax=-95,
        ay=45,
        font={"size": 12, "color": "#333"},
        arrowcolor="#333",
    )
    fig.update_layout(
        title=f"{title}<br><sup>seed dots; diamond/error bars = mean/std over seeds; values aligned to best-test checkpoint step</sup>",
        template="plotly_white",
        height=680,
        width=1200,
        hovermode="closest",
        legend={"orientation": "v", "yanchor": "top", "y": 1, "xanchor": "left", "x": 1.01},
        margin={"l": 80, "r": 340, "t": 105, "b": 70},
    )
    tradeoff_x_range = _plot_range_with_padding(
        pd.concat([
            pd.to_numeric(tradeoff["mean_action0_fraction"], errors="coerce"),
            pd.to_numeric(agg["mean_action0_fraction"], errors="coerce"),
        ], ignore_index=True),
        lower=0,
        upper=1,
        min_span=0.08,
    )
    tradeoff_y_range = _plot_range_with_padding(
        pd.concat([
            pd.to_numeric(tradeoff["survival_pct"], errors="coerce"),
            pd.to_numeric(agg["mean_survival_pct"], errors="coerce"),
        ], ignore_index=True),
        lower=0,
        upper=105,
        min_span=8,
    )
    fig.update_xaxes(title_text="mean action-0 fraction across agents", range=tradeoff_x_range)
    fig.update_yaxes(title_text="episodic survival (%)", range=tradeoff_y_range)

    if SAVE_FIGURES:
        save_path = ctx["FIG_DIR"] / f"{save_name}.html"
        fig.write_html(save_path, include_plotlyjs="cdn")
        print(f"Saved: {save_path}")
    if SHOW_FIGURES:
        fig.show()
    return fig, tradeoff, agg, missing


fig_hvg_tradeoff_heuristic, hvg_tradeoff_heuristic_rows, hvg_tradeoff_heuristic_summary, hvg_tradeoff_heuristic_missing = _plot_checkpoint_action0_survival_tradeoff(
    hvg_checkpoint_action0_by_seed,
    plot_group="heuristic",
    labels=[A0_HVG_BASELINE_LABEL, "global rho heuristic", "local rho heuristic"],
    title=f"A0 HVG: checkpoint-aligned action-0 / survival tradeoff, {A0_HVG_BASELINE_LABEL} vs heuristic overrides",
    save_name="a0_hvg_checkpoint_action0_survival_tradeoff_heuristic",
)

fig_hvg_tradeoff_gate, hvg_tradeoff_gate_rows, hvg_tradeoff_gate_summary, hvg_tradeoff_gate_missing = _plot_checkpoint_action0_survival_tradeoff(
    hvg_checkpoint_action0_by_seed,
    plot_group="gate",
    labels=[A0_HVG_BASELINE_LABEL, "gate final-action MAP", "gate hierarchical greedy"],
    title=f"A0 HVG: checkpoint-aligned action-0 / survival tradeoff, {A0_HVG_BASELINE_LABEL} vs gate variants",
    save_name="a0_hvg_checkpoint_action0_survival_tradeoff_gate",
)

fig_sparse16_tradeoff_flat, sparse16_tradeoff_flat_rows, sparse16_tradeoff_flat_summary, sparse16_tradeoff_flat_missing = _plot_checkpoint_action0_survival_tradeoff(
    sparse16_checkpoint_action0_by_seed,
    plot_group="flat",
    labels=["flat p0.000", "flat p0.001", "flat p0.003", "flat p0.010"],
    title="A0 Sparse16 flat: checkpoint-aligned action-0 / survival tradeoff",
    save_name="a0_sparse16_checkpoint_action0_survival_tradeoff_flat",
)

fig_sparse16_tradeoff_gated, sparse16_tradeoff_gated_rows, sparse16_tradeoff_gated_summary, sparse16_tradeoff_gated_missing = _plot_checkpoint_action0_survival_tradeoff(
    sparse16_checkpoint_action0_by_seed,
    plot_group="gated",
    labels=["gated p0.000", "gated p0.001", "gated p0.003", "gated p0.010"],
    title="A0 Sparse16 gated: checkpoint-aligned action-0 / survival tradeoff",
    save_name="a0_sparse16_checkpoint_action0_survival_tradeoff_gated",
)

fig_aib_tradeoff_flat, aib_tradeoff_flat_rows, aib_tradeoff_flat_summary, aib_tradeoff_flat_missing = _plot_checkpoint_action0_survival_tradeoff(
    aib_checkpoint_action0_by_seed,
    plot_group="flat",
    labels=[A0_HVG_BASELINE_LABEL, "flat local t0.10", "flat local t0.20", "flat local t0.35", "flat non-idle t0.20"],
    title="A0 AIB flat budget variants: checkpoint-aligned action-0 / survival tradeoff",
    save_name="a0_aib_checkpoint_action0_survival_tradeoff_flat",
)

fig_aib_tradeoff_gate, aib_tradeoff_gate_rows, aib_tradeoff_gate_summary, aib_tradeoff_gate_missing = _plot_checkpoint_action0_survival_tradeoff(
    aib_checkpoint_action0_by_seed,
    plot_group="gate",
    labels=[A0_HVG_BASELINE_LABEL, "flat local t0.20", "gate h-greedy t0.20"],
    title="A0 AIB gated budget variant: checkpoint-aligned action-0 / survival tradeoff",
    save_name="a0_aib_checkpoint_action0_survival_tradeoff_gate",
)

COMBINED_TRADEOFF_SOURCES = [
    {
        "tradeoff_family": "HVG heuristic",
        "rows": hvg_tradeoff_heuristic_rows,
        "summary": hvg_tradeoff_heuristic_summary,
    },
    {
        "tradeoff_family": "HVG gate",
        "rows": hvg_tradeoff_gate_rows,
        "summary": hvg_tradeoff_gate_summary,
    },
    {
        "tradeoff_family": "Sparse16 flat",
        "rows": sparse16_tradeoff_flat_rows,
        "summary": sparse16_tradeoff_flat_summary,
    },
    {
        "tradeoff_family": "Sparse16 gated",
        "rows": sparse16_tradeoff_gated_rows,
        "summary": sparse16_tradeoff_gated_summary,
    },
    {
        "tradeoff_family": "AIB flat",
        "rows": aib_tradeoff_flat_rows,
        "summary": aib_tradeoff_flat_summary,
    },
    {
        "tradeoff_family": "AIB gate",
        "rows": aib_tradeoff_gate_rows,
        "summary": aib_tradeoff_gate_summary,
    },
]

combined_seed_frames = []
combined_mean_frames = []
for source in COMBINED_TRADEOFF_SOURCES:
    rows = source["rows"]
    if isinstance(rows, pd.DataFrame) and not rows.empty:
        frame = rows.copy()
        frame["tradeoff_family"] = source["tradeoff_family"]
        frame["variation"] = frame["condition_label"].astype(str)
        combined_seed_frames.append(frame)

    summary = source["summary"]
    if isinstance(summary, pd.DataFrame) and not summary.empty:
        frame = summary.copy()
        frame["tradeoff_family"] = source["tradeoff_family"]
        frame["variation"] = frame["condition_label"].astype(str)
        combined_mean_frames.append(frame)

combined_tradeoff_seed_rows = (
    pd.concat(combined_seed_frames, ignore_index=True, sort=False)
    if combined_seed_frames
    else pd.DataFrame()
)
combined_tradeoff_mean_rows = (
    pd.concat(combined_mean_frames, ignore_index=True, sort=False)
    if combined_mean_frames
    else pd.DataFrame()
)

if combined_tradeoff_seed_rows.empty or combined_tradeoff_mean_rows.empty:
    print("No combined checkpoint-aligned action-0 / survival tradeoff data available.")
else:
    family_order = [source["tradeoff_family"] for source in COMBINED_TRADEOFF_SOURCES]
    family_symbols = {
        "HVG heuristic": "circle",
        "HVG gate": "diamond",
        "Sparse16 flat": "square",
        "Sparse16 gated": "cross",
        "AIB flat": "triangle-up",
        "AIB gate": "x",
    }
    variation_order = list(dict.fromkeys(combined_tradeoff_mean_rows["variation"].astype(str).tolist()))
    palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24 + px.colors.qualitative.Safe
    variation_colors = {
        variation: A0_HVG_VARIANT_COLORS.get(variation, palette[idx % len(palette)])
        for idx, variation in enumerate(variation_order)
    }

    combined_tradeoff_audit = combined_tradeoff_mean_rows.copy().rename(columns={
        "mean_action0_fraction": "condition_mean_plotted_action0_fraction",
        "std_action0_fraction": "condition_std_plotted_action0_fraction",
        "mean_survival_pct": "condition_mean_plotted_survival_pct",
        "std_survival_pct": "condition_std_plotted_survival_pct",
    })
    combined_tradeoff_audit_columns = [
        "tradeoff_family",
        "variation",
        "condition_mean_plotted_action0_fraction",
        "condition_std_plotted_action0_fraction",
        "condition_mean_plotted_survival_pct",
        "condition_std_plotted_survival_pct",
        "n_seeds",
        "seeds",
        "runs",
        "checkpoint_files",
        "survival_metrics",
    ]
    combined_tradeoff_audit_columns = [
        column for column in combined_tradeoff_audit_columns
        if column in combined_tradeoff_audit.columns
    ]
    combined_tradeoff_audit = (
        combined_tradeoff_audit[combined_tradeoff_audit_columns]
        .sort_values(["tradeoff_family", "variation"], kind="stable")
        .reset_index(drop=True)
    )
    if SHOW_CHECKPOINT_AUDIT_TABLES:
        print("Checkpoint tradeoff average values used for: all A0 families combined")
        display(combined_tradeoff_audit)

    fig_combined_checkpoint_action0_survival_tradeoff = go.Figure()
    for family in family_order:
        for variation in variation_order:
            mean_point = combined_tradeoff_mean_rows[
                combined_tradeoff_mean_rows["tradeoff_family"].astype(str).eq(family)
                & combined_tradeoff_mean_rows["variation"].astype(str).eq(variation)
            ]
            if mean_point.empty:
                continue
            mean_row = mean_point.iloc[0]
            color = variation_colors[variation]
            symbol = family_symbols[family]
            fig_combined_checkpoint_action0_survival_tradeoff.add_trace(
                go.Scatter(
                    x=[mean_row["mean_action0_fraction"]],
                    y=[mean_row["mean_survival_pct"]],
                    mode="markers",
                    name=f"{family} | {variation}",
                    legendgroup=f"data:{family}:{variation}",
                    showlegend=False,
                    marker={
                        "color": color,
                        "symbol": symbol,
                        "size": 10,
                        "opacity": 0.95,
                        "line": {"color": "black", "width": 1.1},
                    },
                    customdata=np.array([[
                        family,
                        variation,
                        mean_row["n_seeds"],
                        str(mean_row["seeds"]),
                        str(mean_row["runs"]),
                        str(mean_row["checkpoint_files"]),
                        str(mean_row["survival_metrics"]),
                        mean_row["std_action0_fraction"],
                        mean_row["std_survival_pct"],
                    ]], dtype=object),
                    hovertemplate=(
                        "<b>%{customdata[0]} | %{customdata[1]}</b><br>"
                        "mean action-0=%{x:.4f}<br>"
                        "std action-0=%{customdata[7]:.4f}<br>"
                        "mean survival=%{y:.2f}%<br>"
                        "std survival=%{customdata[8]:.2f}%<br>"
                        "n_seeds=%{customdata[2]}<br>"
                        "seeds=%{customdata[3]}<br>"
                        "runs=%{customdata[4]}<br>"
                        "checkpoints=%{customdata[5]}<br>"
                        "survival metrics=%{customdata[6]}<extra></extra>"
                    ),
                )
            )

    # Compact legend keys: shape encodes the broader run family; color encodes the variation.
    family_legend_labels = {
        "HVG heuristic": "HVG heur.",
        "HVG gate": "HVG gate",
        "Sparse16 flat": "Sparse flat",
        "Sparse16 gated": "Sparse gated",
        "AIB flat": "AIB flat",
        "AIB gate": "AIB gate",
    }
    for family in family_order:
        fig_combined_checkpoint_action0_survival_tradeoff.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                name=family_legend_labels.get(family, family),
                legendgroup="family_shape",
                legendgrouptitle_text="shape = family",
                marker={
                    "symbol": family_symbols[family],
                    "color": "#4a4a4a",
                    "size": 8,
                    "line": {"color": "black", "width": 0.7},
                },
                hoverinfo="skip",
            )
        )
    for variation in variation_order:
        fig_combined_checkpoint_action0_survival_tradeoff.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                name=str(variation),
                legendgroup="variation_color",
                legendgrouptitle_text="color = variation",
                marker={
                    "symbol": "circle",
                    "color": variation_colors[variation],
                    "size": 8,
                    "line": {"color": "black", "width": 0.4},
                },
                hoverinfo="skip",
            )
        )

    fig_combined_checkpoint_action0_survival_tradeoff.add_annotation(
        text="one point = mean over seeds",
        x=0.01,
        y=102,
        xref="x",
        yref="y",
        showarrow=False,
        font={"size": 11, "color": "#333"},
        align="left",
    )
    fig_combined_checkpoint_action0_survival_tradeoff.add_annotation(
        text="better",
        x=0.98,
        y=102,
        xref="x",
        yref="y",
        showarrow=True,
        ax=-55,
        ay=35,
        font={"size": 11, "color": "#333"},
        arrowcolor="#333",
    )
    fig_combined_checkpoint_action0_survival_tradeoff.update_layout(
        title=(
            "All A0 families: mean checkpoint-aligned action-0 / survival tradeoff"
            "<br><sup>shape = run family; color = variation; one point per condition mean</sup>"
        ),
        template="plotly_white",
        height=650,
        width=1180,
        hovermode="closest",
        legend={
            "orientation": "v",
            "yanchor": "top",
            "y": 1,
            "xanchor": "left",
            "x": 1.01,
            "font": {"size": 10},
            "itemsizing": "constant",
        },
        margin={"l": 75, "r": 280, "t": 105, "b": 70},
    )
    combined_x_range = _plot_range_with_padding(
        combined_tradeoff_mean_rows["mean_action0_fraction"],
        lower=0,
        upper=1,
        min_span=0.08,
    )
    combined_y_range = _plot_range_with_padding(
        combined_tradeoff_mean_rows["mean_survival_pct"],
        lower=0,
        upper=105,
        min_span=8,
    )
    fig_combined_checkpoint_action0_survival_tradeoff.update_xaxes(
        title_text="mean action-0 fraction across agents",
        range=combined_x_range,
    )
    fig_combined_checkpoint_action0_survival_tradeoff.update_yaxes(
        title_text="episodic survival (%)",
        range=combined_y_range,
    )

    if SAVE_FIGURES:
        save_path = ctx["FIG_DIR"] / "a0_all_families_checkpoint_action0_survival_tradeoff.html"
        fig_combined_checkpoint_action0_survival_tradeoff.write_html(save_path, include_plotlyjs="cdn")
        print(f"Saved: {save_path}")
    if SHOW_FIGURES:
        fig_combined_checkpoint_action0_survival_tradeoff.show()


print(
    "Teacher-student BC action-0 / survival tradeoff is added below from full-test rollout artifacts."
)


Checkpoint tradeoff values used for: A0 HVG: checkpoint-aligned action-0 / survival tradeoff, a0_hvg_00_baseline vs heuristic overrides


,condition_label,run_name,seed,checkpoint_file,checkpoint_global_step,checkpoint_path,seed_plotted_action0_fraction,seed_plotted_survival_pct,condition_mean_plotted_action0_fraction,condition_mean_plotted_survival_pct,n_seeds,seeds,survival_metric,selected_metric_step,survival_selected_metric_step,step_delta,survival_step_delta
0,a0_hvg_00_baseline,a0_hvg_00_baseline_s0,0,best_test_a0_hvg_00_baseline_s0.tar,13600000,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.250620,100.0,0.435875,100.0,3,"[0, 1, 2]",test/charts/episodic_survival,13600000.0,13600000.0,0.0,0.0
1,a0_hvg_00_baseline,a0_hvg_00_baseline_s1,1,best_test_a0_hvg_00_baseline_s1.tar,13040000,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.469912,100.0,0.435875,100.0,3,"[0, 1, 2]",test/charts/episodic_survival,13040000.0,13040000.0,0.0,0.0
2,a0_hvg_00_baseline,a0_hvg_00_baseline_s2,2,best_test_a0_hvg_00_baseline_s2.tar,13760000,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.587095,100.0,0.435875,100.0,3,"[0, 1, 2]",test/charts/episodic_survival,13760000.0,13760000.0,0.0,0.0
3,global rho heuristic,a0_hvg_01_eval_rho090_s0,0,best_test_a0_hvg_01_eval_rho090_s0.tar,13920000,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.855403,100.0,0.934426,100.0,3,"[0, 1, 2]",test/charts/episodic_survival,13920000.0,13920000.0,0.0,0.0
4,global rho heuristic,a0_hvg_01_eval_rho090_s1,1,best_test_a0_hvg_01_eval_rho090_s1.tar,14320000,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.971251,100.0,0.934426,100.0,3,"[0, 1, 2]",test/charts/episodic_survival,14320000.0,14320000.0,0.0,0.0
5,global rho heuristic,a0_hvg_01_eval_rho090_s2,2,best_test_a0_hvg_01_eval_rho090_s2.tar,14800000,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.976625,100.0,0.934426,100.0,3,"[0, 1, 2]",test/charts/episodic_survival,14800000.0,14800000.0,0.0,0.0
6,local rho heuristic,a0_hvg_04_eval_local_rho090_s0,0,best_test_a0_hvg_04_eval_local_rho090_s0.tar,14160000,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.937202,100.0,0.968316,100.0,3,"[0, 1, 2]",test/charts/episodic_survival,14160000.0,14160000.0,0.0,0.0
7,local rho heuristic,a0_hvg_04_eval_local_rho090_s1,1,best_test_a0_hvg_04_eval_local_rho090_s1.tar,14400000,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.984962,100.0,0.968316,100.0,3,"[0, 1, 2]",test/charts/episodic_survival,14400000.0,14400000.0,0.0,0.0
8,local rho heuristic,a0_hvg_04_eval_local_rho090_s2,2,best_test_a0_hvg_04_eval_local_rho090_s2.tar,14160000,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.982784,100.0,0.968316,100.0,3,"[0, 1, 2]",test/charts/episodic_survival,14160000.0,14160000.0,0.0,0.0


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/a0_hvg_checkpoint_action0_survival_tradeoff_heuristic.html


Checkpoint tradeoff values used for: A0 HVG: checkpoint-aligned action-0 / survival tradeoff, a0_hvg_00_baseline vs gate variants


,condition_label,run_name,seed,checkpoint_file,checkpoint_global_step,checkpoint_path,seed_plotted_action0_fraction,seed_plotted_survival_pct,condition_mean_plotted_action0_fraction,condition_mean_plotted_survival_pct,n_seeds,seeds,survival_metric,selected_metric_step,survival_selected_metric_step,step_delta,survival_step_delta
0,a0_hvg_00_baseline,a0_hvg_00_baseline_s0,0,best_test_a0_hvg_00_baseline_s0.tar,13600000,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.250620,100.000000,0.435875,100.000000,3,"[0, 1, 2]",test/charts/episodic_survival,13600000.0,13600000.0,0.0,0.0
1,a0_hvg_00_baseline,a0_hvg_00_baseline_s1,1,best_test_a0_hvg_00_baseline_s1.tar,13040000,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.469912,100.000000,0.435875,100.000000,3,"[0, 1, 2]",test/charts/episodic_survival,13040000.0,13040000.0,0.0,0.0
2,a0_hvg_00_baseline,a0_hvg_00_baseline_s2,2,best_test_a0_hvg_00_baseline_s2.tar,13760000,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.587095,100.000000,0.435875,100.000000,3,"[0, 1, 2]",test/charts/episodic_survival,13760000.0,13760000.0,0.0,0.0
3,gate final-action MAP,a0_hvg_02_gate_final_map_s0,0,best_test_a0_hvg_02_gate_final_map_s0.tar,12000000,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.774809,96.541419,0.799830,98.155589,3,"[0, 1, 2]",test/charts/episodic_survival,12000000.0,12000000.0,0.0,0.0
4,gate final-action MAP,a0_hvg_02_gate_final_map_s1,1,best_test_a0_hvg_02_gate_final_map_s1.tar,12720000,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.798760,97.925347,0.799830,98.155589,3,"[0, 1, 2]",test/charts/episodic_survival,12720000.0,12720000.0,0.0,0.0
5,gate final-action MAP,a0_hvg_02_gate_final_map_s2,2,best_test_a0_hvg_02_gate_final_map_s2.tar,13760000,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.825922,100.000000,0.799830,98.155589,3,"[0, 1, 2]",test/charts/episodic_survival,13760000.0,13760000.0,0.0,0.0
6,gate hierarchical greedy,a0_hvg_03_gate_hierarchical_s0,0,best_test_a0_hvg_03_gate_hierarchical_s0.tar,7120000,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.406510,100.000000,0.401108,100.000000,3,"[0, 1, 2]",test/charts/episodic_survival,7120000.0,7120000.0,0.0,0.0
7,gate hierarchical greedy,a0_hvg_03_gate_hierarchical_s1,1,best_test_a0_hvg_03_gate_hierarchical_s1.tar,14720000,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.279158,100.000000,0.401108,100.000000,3,"[0, 1, 2]",test/charts/episodic_survival,14720000.0,14720000.0,0.0,0.0
8,gate hierarchical greedy,a0_hvg_03_gate_hierarchical_s2,2,best_test_a0_hvg_03_gate_hierarchical_s2.tar,13760000,/Users/corentinplumet/Documents/RL_Marl2grid/T...,0.517655,100.000000,0.401108,100.000000,3,"[0, 1, 2]",test/charts/episodic_survival,13760000.0,13760000.0,0.0,0.0


...

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/a0_aib_checkpoint_action0_survival_tradeoff_gate.html


Checkpoint tradeoff average values used for: all A0 families combined


,tradeoff_family,variation,condition_mean_plotted_action0_fraction,condition_std_plotted_action0_fraction,condition_mean_plotted_survival_pct,condition_std_plotted_survival_pct,n_seeds,seeds,runs,checkpoint_files,survival_metrics
0,AIB flat,a0_hvg_00_baseline,0.435875,0.170800,100.000000,0.000000,3,"[0, 1, 2]","[a0_hvg_00_baseline_s0, a0_hvg_00_baseline_s1,...","[best_test_a0_hvg_00_baseline_s0.tar, best_tes...",[test/charts/episodic_survival]
1,AIB flat,flat local t0.10,0.977308,0.005742,100.000000,0.000000,3,"[0, 1, 2]","[a0_aib_01_flat_local_t010_s0, a0_aib_01_flat_...","[best_test_a0_aib_01_flat_local_t010_s0.tar, b...",[test/charts/episodic_survival]
2,AIB flat,flat local t0.20,0.979650,0.002457,100.000000,0.000000,3,"[0, 1, 2]","[a0_aib_00_flat_local_t020_s0, a0_aib_00_flat_...","[best_test_a0_aib_00_flat_local_t020_s0.tar, b...",[test/charts/episodic_survival]
3,AIB flat,flat local t0.35,0.934073,0.022588,100.000000,0.000000,3,"[0, 1, 2]","[a0_aib_02_flat_local_t035_s0, a0_aib_02_flat_...","[best_test_a0_aib_02_flat_local_t035_s0.tar, b...",[test/charts/episodic_survival]
4,AIB flat,flat non-idle t0.20,0.985926,0.006870,99.616815,0.663695,3,"[0, 1, 2]","[a0_aib_04_flat_nonidle_t020_s0, a0_aib_04_fla...","[best_test_a0_aib_04_flat_nonidle_t020_s0.tar,...",[test/charts/episodic_survival]
5,AIB gate,a0_hvg_00_baseline,0.435875,0.170800,100.000000,0.000000,3,"[0, 1, 2]","[a0_hvg_00_baseline_s0, a0_hvg_00_baseline_s1,...","[best_test_a0_hvg_00_baseline_s0.tar, best_tes...",[test/charts/episodic_survival]
6,AIB gate,flat local t0.20,0.979650,0.002457,100.000000,0.000000,3,"[0, 1, 2]","[a0_aib_00_flat_local_t020_s0, a0_aib_00_flat_...","[best_test_a0_aib_00_flat_local_t020_s0.tar, b...",[test/charts/episodic_survival]
7,AIB gate,gate h-greedy t0.20,0.999447,0.000505,50.374091,22.056380,3,"[0, 1, 2]","[a0_aib_03_gate_hgreedy_sep_local_t020_s0, a0_...",[best_test_a0_aib_03_gate_hgreedy_sep_local_t0...,[test/charts/episodic_survival]
8,HVG gate,a0_hvg_00_baseline,0.435875,0.170800,100.000000,0.000000,3,"[0, 1, 2]","[a0_hvg_00_baseline_s0, a0_hvg_00_baseline_s1,...","[best_test_a0_hvg_00_baseline_s0.tar, best_tes...",[test/charts/episodic_survival]
9,HVG gate,gate final-action MAP,0.799830,0.025573,98.155589,1.740748,3,"[0, 1, 2]","[a0_hvg_02_gate_final_map_s0, a0_hvg_02_gate_f...","[best_test_a0_hvg_02_gate_final_map_s0.tar, be...",[test/charts/episodic_survival]


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/a0_all_families_checkpoint_action0_survival_tradeoff.html


Teacher-student BC action-0 / survival tradeoff is added below from full-test rollout artifacts.


## Selected Action-0 / Survival Tradeoff

Focused slide-ready comparison with only the requested conditions: flat baseline, rho heuristics, intervention gates, flat intervention penalties, AIB flat local 0.10/0.20, and the two imitation-learning runs. Action-0 stays checkpoint-aligned, while survival uses the saved full-test rollout artifacts for every condition.


In [29]:
import json

FOCUSED_TRADEOFF_LABEL_RENAME = {
    A0_HVG_BASELINE_LABEL: 'baseline flat',
    'gate final-action MAP': 'intervention gate MAP',
    'gate hierarchical greedy': 'intervention gate hierarchical',
    'flat p0.000': 'flat intervention penalty 0.000',
    'flat p0.001': 'flat intervention penalty 0.001',
    'flat p0.003': 'flat intervention penalty 0.003',
    'flat p0.010': 'flat intervention penalty 0.010',
    'flat local t0.10': 'AIB flat local 0.10',
    'flat local t0.20': 'AIB flat local 0.20',
}

FOCUSED_TRADEOFF_LABEL_ORDER = [
    'baseline flat',
    'local rho heuristic',
    'global rho heuristic',
    'intervention gate MAP',
    'intervention gate hierarchical',
    'flat intervention penalty 0.000',
    'flat intervention penalty 0.001',
    'flat intervention penalty 0.003',
    'flat intervention penalty 0.010',
    'AIB flat local 0.10',
    'AIB flat local 0.20',
    TEACHER_STUDENT_FAMILY_LABELS['local_bc'],
    TEACHER_STUDENT_FAMILY_LABELS['local_bc_bal020_w3_aux025'],
]
FOCUSED_TRADEOFF_ORDER = {label: idx for idx, label in enumerate(FOCUSED_TRADEOFF_LABEL_ORDER)}

FOCUSED_TRADEOFF_FAMILY_BY_LABEL = {
    'baseline flat': 'baseline',
    'local rho heuristic': 'rho heuristic',
    'global rho heuristic': 'rho heuristic',
    'intervention gate MAP': 'intervention gate',
    'intervention gate hierarchical': 'intervention gate',
    'flat intervention penalty 0.000': 'flat intervention penalty',
    'flat intervention penalty 0.001': 'flat intervention penalty',
    'flat intervention penalty 0.003': 'flat intervention penalty',
    'flat intervention penalty 0.010': 'flat intervention penalty',
    'AIB flat local 0.10': 'AIB flat',
    'AIB flat local 0.20': 'AIB flat',
    TEACHER_STUDENT_FAMILY_LABELS['local_bc']: 'imitation learning',
    TEACHER_STUDENT_FAMILY_LABELS['local_bc_bal020_w3_aux025']: 'imitation learning',
}

FOCUSED_TRADEOFF_COLORS = {
    'baseline flat': '#1f77b4',
    'local rho heuristic': '#2ca02c',
    'global rho heuristic': '#ff7f0e',
    'intervention gate MAP': '#d62728',
    'intervention gate hierarchical': '#9467bd',
    'flat intervention penalty 0.000': '#8c564b',
    'flat intervention penalty 0.001': '#e377c2',
    'flat intervention penalty 0.003': '#7f7f7f',
    'flat intervention penalty 0.010': '#bcbd22',
    'AIB flat local 0.10': '#17becf',
    'AIB flat local 0.20': '#003f5c',
    TEACHER_STUDENT_FAMILY_LABELS['local_bc']: TEACHER_STUDENT_COLORS[TEACHER_STUDENT_FAMILY_LABELS['local_bc']],
    TEACHER_STUDENT_FAMILY_LABELS['local_bc_bal020_w3_aux025']: TEACHER_STUDENT_COLORS[TEACHER_STUDENT_FAMILY_LABELS['local_bc_bal020_w3_aux025']],
}
A0_HVG_VARIANT_COLORS.update(FOCUSED_TRADEOFF_COLORS)


def _focused_checkpoint_tradeoff_rows(source_rows, *, labels, plot_group, family_label):
    tradeoff, _summary, missing = _checkpoint_action0_survival_tradeoff_rows(
        source_rows,
        labels=labels,
        plot_group=plot_group,
    )
    if not missing.empty:
        print(f'{family_label}: missing survival for {len(missing)} checkpoint/run rows.')
        if SHOW_DIAGNOSTIC_TABLES:
            display(missing)
    if tradeoff.empty:
        return pd.DataFrame()

    rows = tradeoff.copy()
    rows['source_condition_label'] = rows['condition_label'].astype(str)
    rows['condition_label'] = rows['source_condition_label'].replace(FOCUSED_TRADEOFF_LABEL_RENAME)
    rows['variation'] = rows['condition_label'].astype(str)
    rows['tradeoff_family'] = rows['variation'].map(FOCUSED_TRADEOFF_FAMILY_BY_LABEL).fillna(family_label)
    rows['label_order'] = rows['variation'].map(FOCUSED_TRADEOFF_ORDER).fillna(999).astype(int)
    rows = _apply_full_test_survival(rows, family_label=family_label)
    return rows


def _apply_full_test_survival(rows, *, family_label):
    if rows is None or rows.empty:
        return pd.DataFrame()
    if 'a0_full_test' not in globals() or a0_full_test.empty:
        print(f'{family_label}: no full-test eval table available; keeping W&B checkpoint survival.')
        return rows

    lookup_columns = [
        'run_like',
        'survival_percent',
        'survival_frac',
        'checkpoint_global_step',
        'eval_episodes',
        'path',
    ]
    lookup_columns = [column for column in lookup_columns if column in a0_full_test.columns]
    lookup = a0_full_test[lookup_columns].dropna(subset=['run_like']).copy()
    if lookup.empty:
        print(f'{family_label}: no full-test rows with run_like available; keeping W&B checkpoint survival.')
        return rows

    lookup = lookup.sort_values(['run_like', 'checkpoint_global_step'], kind='stable')
    duplicate_run_likes = lookup['run_like'][lookup['run_like'].duplicated()].drop_duplicates().astype(str).tolist()
    if duplicate_run_likes:
        print(f'{family_label}: duplicate full-test rows for {duplicate_run_likes}; keeping the last checkpoint row per run.')
    lookup = lookup.drop_duplicates('run_like', keep='last').rename(columns={
        'survival_percent': 'full_test_survival_percent',
        'survival_frac': 'full_test_survival_frac',
        'checkpoint_global_step': 'full_test_checkpoint_global_step',
        'eval_episodes': 'full_test_eval_episodes',
        'path': 'full_test_eval_json',
    })

    merged = rows.merge(lookup, left_on='run_name', right_on='run_like', how='left')
    matched = pd.to_numeric(merged.get('full_test_survival_percent'), errors='coerce').notna()
    if not matched.all():
        missing_runs = sorted(merged.loc[~matched, 'run_name'].dropna().astype(str).unique().tolist())
        if missing_runs:
            print(f'{family_label}: no full-test survival for {missing_runs}; keeping W&B checkpoint survival for those rows.')

    merged.loc[matched, 'survival_pct'] = pd.to_numeric(
        merged.loc[matched, 'full_test_survival_percent'],
        errors='coerce',
    )
    merged.loc[matched, 'survival_metric'] = 'full_test_eval.survival_percent'
    merged.loc[matched, 'survival_selected_metric_step'] = pd.to_numeric(
        merged.loc[matched, 'full_test_checkpoint_global_step'],
        errors='coerce',
    )
    merged.loc[matched, 'survival_step_delta'] = 0
    return merged


def _resolve_action_summary_path(eval_payload, eval_stem):
    artifacts = eval_payload.get('action_artifacts') or {}
    action_summary = artifacts.get('action_summary_json')
    candidates = []
    if action_summary:
        action_path = Path(action_summary)
        candidates.append(action_path if action_path.is_absolute() else TASK_DIR / action_path)
    candidates.append(TASK_DIR / 'outputs' / 'full_test_eval_actions' / eval_stem / 'action_summary.json')
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Could not find action_summary.json for {eval_stem}; tried {candidates}')


def _load_teacher_student_full_test_tradeoff_rows():
    eval_dir = TASK_DIR / 'outputs' / 'full_test_eval'
    if not eval_dir.exists():
        print(f'Teacher-student full-test eval directory not found: {eval_dir}')
        return pd.DataFrame()

    family_specs = [
        ('local_bc', TEACHER_STUDENT_FAMILY_LABELS['local_bc'], 200),
        ('local_bc_bal020_w3_aux025', TEACHER_STUDENT_FAMILY_LABELS['local_bc_bal020_w3_aux025'], 201),
    ]
    rows = []
    for family, label, order in family_specs:
        for eval_path in sorted(eval_dir.glob(f'{family}_s*_step*_job*.json')):
            match = re.match(
                rf'^{re.escape(family)}_s(?P<seed>\d+)_step(?P<step>\d+)_job(?P<job>\d+)$',
                eval_path.stem,
            )
            if match is None:
                continue
            eval_payload = json.loads(eval_path.read_text())
            action_path = _resolve_action_summary_path(eval_payload, eval_path.stem)
            action_payload = json.loads(action_path.read_text())
            agent_rows = action_payload.get('agents') or {}
            agent_action0 = []
            agent_names = []
            for agent, stats in sorted(agent_rows.items()):
                value = pd.to_numeric(stats.get('action0_fraction'), errors='coerce')
                if pd.notna(value):
                    agent_names.append(str(agent))
                    agent_action0.append(float(value))
            if not agent_action0:
                print(f'No per-agent action0_fraction values found in {action_path}')
                continue

            survival_pct = eval_payload.get('survival_percent')
            if survival_pct is None and eval_payload.get('survival_frac') is not None:
                survival_pct = float(eval_payload['survival_frac']) * 100.0
            if survival_pct is None:
                print(f'No survival_percent/survival_frac found in {eval_path}')
                continue

            checkpoint_path = str(eval_payload.get('checkpoint') or '')
            checkpoint_global_step = eval_payload.get('checkpoint_global_step', action_payload.get('global_step', match.group('step')))
            rows.append({
                'plot_group': 'imitation_learning',
                'condition_label': label,
                'source_condition_label': family,
                'variation': label,
                'tradeoff_family': FOCUSED_TRADEOFF_FAMILY_BY_LABEL[label],
                'order': order,
                'label_order': FOCUSED_TRADEOFF_ORDER[label],
                'run_name': f'{family}_s{int(match.group("seed"))}',
                'run_id': f'{family}_s{int(match.group("seed"))}',
                'seed': int(match.group('seed')),
                'checkpoint_path': checkpoint_path,
                'checkpoint_file': Path(checkpoint_path).name if checkpoint_path else '',
                'checkpoint_global_step': int(checkpoint_global_step),
                'selected_metric_step': int(action_payload.get('global_step', match.group('step'))),
                'survival_selected_metric_step': int(checkpoint_global_step),
                'step_delta': 0,
                'survival_step_delta': 0,
                'step_match_policy': 'full_test_eval',
                'mean_action0_fraction': float(pd.Series(agent_action0).mean()),
                'std_agent_action0_fraction': float(pd.Series(agent_action0).std(ddof=1)) if len(agent_action0) > 1 else 0.0,
                'n_agents': len(agent_action0),
                'agents': agent_names,
                'survival_metric': 'full_test_eval.survival_percent',
                'survival_pct': float(survival_pct),
                'action_summary_json': str(action_path),
                'full_test_eval_json': str(eval_path),
                'eval_episodes': int(eval_payload.get('eval_episodes', action_payload.get('eval_episodes', 0))),
            })
    return pd.DataFrame(rows)


def _focused_tradeoff_summary(seed_rows):
    if seed_rows is None or seed_rows.empty:
        return pd.DataFrame()
    summary = (
        seed_rows
        .groupby(['tradeoff_family', 'variation', 'label_order'], dropna=False, as_index=False)
        .agg(
            mean_action0_fraction=('mean_action0_fraction', 'mean'),
            std_action0_fraction=('mean_action0_fraction', 'std'),
            mean_survival_pct=('survival_pct', 'mean'),
            std_survival_pct=('survival_pct', 'std'),
            n_seeds=('seed', 'nunique'),
            seeds=('seed', lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
            runs=('run_name', lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
            checkpoint_files=('checkpoint_file', lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
            survival_metrics=('survival_metric', lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
        )
        .sort_values('label_order', kind='stable')
        .reset_index(drop=True)
    )
    summary['std_action0_fraction'] = summary['std_action0_fraction'].fillna(0.0)
    summary['std_survival_pct'] = summary['std_survival_pct'].fillna(0.0)
    return summary


focused_checkpoint_frames = [
    _focused_checkpoint_tradeoff_rows(
        hvg_checkpoint_action0_by_seed,
        plot_group=None,
        labels=[
            A0_HVG_BASELINE_LABEL,
            'local rho heuristic',
            'global rho heuristic',
            'gate final-action MAP',
            'gate hierarchical greedy',
        ],
        family_label='HVG selected',
    ),
    _focused_checkpoint_tradeoff_rows(
        sparse16_checkpoint_action0_by_seed,
        plot_group='flat',
        labels=['flat p0.000', 'flat p0.001', 'flat p0.003', 'flat p0.010'],
        family_label='flat intervention penalty',
    ),
    _focused_checkpoint_tradeoff_rows(
        aib_checkpoint_action0_by_seed,
        plot_group='flat',
        labels=['flat local t0.10', 'flat local t0.20'],
        family_label='AIB flat',
    ),
]
focused_imitation_tradeoff_rows = _load_teacher_student_full_test_tradeoff_rows()
focused_tradeoff_seed_rows = pd.concat(
    [frame for frame in [*focused_checkpoint_frames, focused_imitation_tradeoff_rows] if isinstance(frame, pd.DataFrame) and not frame.empty],
    ignore_index=True,
    sort=False,
) if any(isinstance(frame, pd.DataFrame) and not frame.empty for frame in [*focused_checkpoint_frames, focused_imitation_tradeoff_rows]) else pd.DataFrame()

if focused_tradeoff_seed_rows.empty:
    print('No selected action-0 / survival tradeoff data available.')
else:
    focused_tradeoff_seed_rows['variation'] = focused_tradeoff_seed_rows['condition_label'].astype(str)
    focused_tradeoff_seed_rows['tradeoff_family'] = focused_tradeoff_seed_rows['variation'].map(
        FOCUSED_TRADEOFF_FAMILY_BY_LABEL
    ).fillna(focused_tradeoff_seed_rows['tradeoff_family'])
    focused_tradeoff_seed_rows['label_order'] = focused_tradeoff_seed_rows['variation'].map(FOCUSED_TRADEOFF_ORDER).fillna(999).astype(int)
    focused_tradeoff_seed_rows['agents_display'] = focused_tradeoff_seed_rows['agents'].map(
        lambda values: ', '.join(values) if isinstance(values, (list, tuple)) else str(values)
    )
    focused_tradeoff_seed_rows = focused_tradeoff_seed_rows.sort_values(
        ['label_order', 'seed', 'run_name'],
        kind='stable',
    ).reset_index(drop=True)
    focused_tradeoff_summary = _focused_tradeoff_summary(focused_tradeoff_seed_rows)

    if SHOW_CHECKPOINT_AUDIT_TABLES:
        focused_audit_columns = [
            'tradeoff_family',
            'variation',
            'run_name',
            'seed',
            'checkpoint_file',
            'checkpoint_global_step',
            'selected_metric_step',
            'mean_action0_fraction',
            'std_agent_action0_fraction',
            'survival_pct',
            'survival_metric',
            'agents_display',
        ]
        focused_audit_columns = [column for column in focused_audit_columns if column in focused_tradeoff_seed_rows.columns]
        print('Selected action-0 / survival tradeoff seed values')
        display(focused_tradeoff_seed_rows[focused_audit_columns])

        focused_summary_display = focused_tradeoff_summary.rename(columns={
            'variation': 'condition',
            'mean_action0_fraction': 'condition_mean_action0_fraction',
            'std_action0_fraction': 'condition_std_action0_fraction',
            'mean_survival_pct': 'condition_mean_survival_pct',
            'std_survival_pct': 'condition_std_survival_pct',
        })
        print('Selected action-0 / survival tradeoff condition means')
        display(focused_summary_display[[
            'tradeoff_family',
            'condition',
            'condition_mean_action0_fraction',
            'condition_std_action0_fraction',
            'condition_mean_survival_pct',
            'condition_std_survival_pct',
            'n_seeds',
            'seeds',
            'runs',
            'checkpoint_files',
            'survival_metrics',
        ]])

    focused_family_symbols = {
        'baseline': 'circle',
        'rho heuristic': 'circle-open',
        'intervention gate': 'diamond',
        'flat intervention penalty': 'square',
        'AIB flat': 'triangle-up',
        'imitation learning': 'star',
    }
    focused_label_order = [label for label in FOCUSED_TRADEOFF_LABEL_ORDER if label in set(focused_tradeoff_summary['variation'].astype(str))]
    fallback_palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24 + px.colors.qualitative.Safe
    focused_colors = {
        label: FOCUSED_TRADEOFF_COLORS.get(label, fallback_palette[idx % len(fallback_palette)])
        for idx, label in enumerate(focused_label_order)
    }

    fig_selected_action0_survival_tradeoff = go.Figure()
    for label in focused_label_order:
        seed_points = focused_tradeoff_seed_rows[focused_tradeoff_seed_rows['variation'].astype(str).eq(label)].copy()
        mean_point = focused_tradeoff_summary[focused_tradeoff_summary['variation'].astype(str).eq(label)]
        if seed_points.empty or mean_point.empty:
            continue
        family = str(mean_point.iloc[0]['tradeoff_family'])
        color = focused_colors[label]
        symbol = focused_family_symbols.get(family, 'circle')
        fig_selected_action0_survival_tradeoff.add_trace(
            go.Scatter(
                x=seed_points['mean_action0_fraction'],
                y=seed_points['survival_pct'],
                mode='markers',
                name=f'{label} seeds',
                legendgroup=label,
                showlegend=False,
                marker={
                    'color': color,
                    'symbol': symbol,
                    'size': 6,
                    'opacity': 0.45,
                    'line': {'color': 'white', 'width': 0.9},
                },
                customdata=seed_points[[
                    'tradeoff_family',
                    'variation',
                    'run_name',
                    'seed',
                    'checkpoint_file',
                    'checkpoint_global_step',
                    'survival_metric',
                    'agents_display',
                ]].astype(str).to_numpy(),
                hovertemplate=(
                    '<b>%{customdata[1]}</b><br>'
                    'family=%{customdata[0]}<br>'
                    'run=%{customdata[2]}<br>'
                    'seed=%{customdata[3]}<br>'
                    'checkpoint=%{customdata[4]}<br>'
                    'checkpoint_step=%{customdata[5]}<br>'
                    'mean action-0=%{x:.4f}<br>'
                    'survival=%{y:.2f}%<br>'
                    'survival metric=%{customdata[6]}<br>'
                    'agents=%{customdata[7]}<extra></extra>'
                ),
            )
        )

        mean_row = mean_point.iloc[0]
        fig_selected_action0_survival_tradeoff.add_trace(
            go.Scatter(
                x=[mean_row['mean_action0_fraction']],
                y=[mean_row['mean_survival_pct']],
                mode='markers',
                name=label,
                legendgroup=label,
                showlegend=True,
                marker={
                    'color': color,
                    'symbol': symbol,
                    'size': 13,
                    'opacity': 0.98,
                    'line': {'color': 'black', 'width': 1.2},
                },
                customdata=np.array([[
                    family,
                    label,
                    mean_row['n_seeds'],
                    str(mean_row['seeds']),
                    str(mean_row['runs']),
                    str(mean_row['checkpoint_files']),
                    str(mean_row['survival_metrics']),
                    mean_row['std_action0_fraction'],
                    mean_row['std_survival_pct'],
                ]], dtype=object),
                hovertemplate=(
                    '<b>%{customdata[1]}</b><br>'
                    'family=%{customdata[0]}<br>'
                    'mean action-0=%{x:.4f}<br>'
                    'std action-0=%{customdata[7]:.4f}<br>'
                    'mean survival=%{y:.2f}%<br>'
                    'std survival=%{customdata[8]:.2f}%<br>'
                    'n_seeds=%{customdata[2]}<br>'
                    'seeds=%{customdata[3]}<br>'
                    'runs=%{customdata[4]}<br>'
                    'checkpoints=%{customdata[5]}<br>'
                    'survival metrics=%{customdata[6]}<extra></extra>'
                ),
            )
        )

    fig_selected_action0_survival_tradeoff.add_annotation(
        text='small markers = seeds; large markers = condition means',
        x=0.01,
        y=102,
        xref='x',
        yref='y',
        showarrow=False,
        font={'size': 11, 'color': '#333'},
        align='left',
    )
    # fig_selected_action0_survival_tradeoff.add_annotation(
    #     text='better',
    #     x=0.995,
    #     y=102,
    #     xref='x',
    #     yref='y',
    #     showarrow=True,
    #     ax=-55,
    #     ay=35,
    #     font={'size': 11, 'color': '#333'},
    #     arrowcolor='#333',
    # )
    fig_selected_action0_survival_tradeoff.update_layout(
        title=(
            'Selected A0 models: action-0 / survival tradeoff'
            '<br><sup>action-0 is checkpoint-aligned; survival comes from saved full-test rollout artifacts</sup>'
        ),
        template='plotly_white',
        height=680,
        width=1180,
        hovermode='closest',
        legend={
            'orientation': 'v',
            'yanchor': 'top',
            'y': 1,
            'xanchor': 'left',
            'x': 1.01,
            'font': {'size': 10},
            'itemsizing': 'constant',
        },
        margin={'l': 75, 'r': 310, 't': 105, 'b': 70},
    )
    selected_x_range = _plot_range_with_padding(
        pd.concat([
            pd.to_numeric(focused_tradeoff_seed_rows['mean_action0_fraction'], errors='coerce'),
            pd.to_numeric(focused_tradeoff_summary['mean_action0_fraction'], errors='coerce'),
        ], ignore_index=True),
        lower=0,
        upper=1,
        min_span=0.08,
    )
    selected_y_range = _plot_range_with_padding(
        pd.concat([
            pd.to_numeric(focused_tradeoff_seed_rows['survival_pct'], errors='coerce'),
            pd.to_numeric(focused_tradeoff_summary['mean_survival_pct'], errors='coerce'),
        ], ignore_index=True),
        lower=0,
        upper=105,
        min_span=8,
    )
    fig_selected_action0_survival_tradeoff.update_xaxes(
        title_text='mean action-0 fraction across agents',
        range=selected_x_range,
    )
    fig_selected_action0_survival_tradeoff.update_yaxes(
        title_text='episodic survival (%)',
        range=selected_y_range,
    )

    if SAVE_FIGURES:
        save_path = ctx['FIG_DIR'] / 'a0_selected_action0_survival_tradeoff.html'
        fig_selected_action0_survival_tradeoff.write_html(save_path, include_plotlyjs='cdn')
        print(f'Saved: {save_path}')
    if SHOW_FIGURES:
        fig_selected_action0_survival_tradeoff.show()


Selected action-0 / survival tradeoff seed values


,tradeoff_family,variation,run_name,seed,checkpoint_file,checkpoint_global_step,selected_metric_step,mean_action0_fraction,std_agent_action0_fraction,survival_pct,survival_metric,agents_display
0,baseline,baseline flat,a0_hvg_00_baseline_s0,0,best_test_a0_hvg_00_baseline_s0.tar,13600000,13600000.0,0.250620,0.090884,95.322927,full_test_eval.survival_percent,"agent_0, agent_1, agent_2"
1,baseline,baseline flat,a0_hvg_00_baseline_s1,1,best_test_a0_hvg_00_baseline_s1.tar,13040000,13040000.0,0.469912,0.019882,91.455791,full_test_eval.survival_percent,"agent_0, agent_1, agent_2"
2,baseline,baseline flat,a0_hvg_00_baseline_s2,2,best_test_a0_hvg_00_baseline_s2.tar,13760000,13760000.0,0.587095,0.299012,94.216541,full_test_eval.survival_percent,"agent_0, agent_1, agent_2"
3,rho heuristic,local rho heuristic,a0_hvg_04_eval_local_rho090_s0,0,best_test_a0_hvg_04_eval_local_rho090_s0.tar,14160000,14160000.0,0.937202,0.075944,99.142741,full_test_eval.survival_percent,"agent_0, agent_1, agent_2"
4,rho heuristic,local rho heuristic,a0_hvg_04_eval_local_rho090_s1,1,best_test_a0_hvg_04_eval_local_rho090_s1.tar,14400000,14400000.0,0.984962,0.012689,97.407309,full_test_eval.survival_percent,"agent_0, agent_1, agent_2"
5,rho heuristic,local rho heuristic,a0_hvg_04_eval_local_rho090_s2,2,best_test_a0_hvg_04_eval_local_rho090_s2.tar,14160000,14160000.0,0.982784,0.026809,94.670682,full_test_eval.survival_percent,"agent_0, agent_1, agent_2"
6,rho heuristic,global rho heuristic,a0_hvg_01_eval_rho090_s0,0,best_test_a0_hvg_01_eval_rho090_s0.tar,13920000,13920000.0,0.855403,0.022499,98.309235,full_test_eval.survival_percent,"agent_0, agent_1, agent_2"
7,rho heuristic,global rho heuristic,a0_hvg_01_eval_rho090_s1,1,best_test_a0_hvg_01_eval_rho090_s1.tar,14320000,14320000.0,0.971251,0.021610,99.655616,full_test_eval.survival_percent,"agent_0, agent_1, agent_2"
8,rho heuristic,global rho heuristic,a0_hvg_01_eval_rho090_s2,2,best_test_a0_hvg_01_eval_rho090_s2.tar,14800000,14800000.0,0.976625,0.016614,97.100250,full_test_eval.survival_percent,"agent_0, agent_1, agent_2"
9,intervention gate,intervention gate MAP,a0_hvg_02_gate_final_map_s0,0,best_test_a0_hvg_02_gate_final_map_s0.tar,12000000,12000000.0,0.774809,0.092940,83.708380,full_test_eval.survival_percent,"agent_0, agent_1, agent_2"


Selected action-0 / survival tradeoff condition means


,tradeoff_family,condition,condition_mean_action0_fraction,condition_std_action0_fraction,condition_mean_survival_pct,condition_std_survival_pct,n_seeds,seeds,runs,checkpoint_files,survival_metrics
0,baseline,baseline flat,0.435875,0.170800,93.665086,1.991673,3,"[0, 1, 2]","[a0_hvg_00_baseline_s0, a0_hvg_00_baseline_s1,...","[best_test_a0_hvg_00_baseline_s0.tar, best_tes...",[full_test_eval.survival_percent]
1,rho heuristic,local rho heuristic,0.968316,0.026967,97.073577,2.254631,3,"[0, 1, 2]","[a0_hvg_04_eval_local_rho090_s0, a0_hvg_04_eva...","[best_test_a0_hvg_04_eval_local_rho090_s0.tar,...",[full_test_eval.survival_percent]
2,rho heuristic,global rho heuristic,0.934426,0.068489,98.355034,1.278298,3,"[0, 1, 2]","[a0_hvg_01_eval_rho090_s0, a0_hvg_01_eval_rho0...","[best_test_a0_hvg_01_eval_rho090_s0.tar, best_...",[full_test_eval.survival_percent]
3,intervention gate,intervention gate MAP,0.799830,0.025573,89.817159,5.307445,3,"[0, 1, 2]","[a0_hvg_02_gate_final_map_s0, a0_hvg_02_gate_f...","[best_test_a0_hvg_02_gate_final_map_s0.tar, be...",[full_test_eval.survival_percent]
4,intervention gate,intervention gate hierarchical,0.401108,0.119340,93.655153,3.714422,3,"[0, 1, 2]","[a0_hvg_03_gate_hierarchical_s0, a0_hvg_03_gat...","[best_test_a0_hvg_03_gate_hierarchical_s0.tar,...",[full_test_eval.survival_percent]
5,flat intervention penalty,flat intervention penalty 0.001,0.488681,0.101384,96.752041,0.490607,3,"[0, 1, 2]","[a0_sparse16_flat_p001_s0, a0_sparse16_flat_p0...","[best_test_a0_sparse16_flat_p001_s0.tar, best_...",[full_test_eval.survival_percent]
6,flat intervention penalty,flat intervention penalty 0.003,0.577582,0.122331,93.969245,2.843161,3,"[0, 1, 2]","[a0_sparse16_flat_p003_s0, a0_sparse16_flat_p0...","[best_test_a0_sparse16_flat_p003_s0.tar, best_...",[full_test_eval.survival_percent]
7,flat intervention penalty,flat intervention penalty 0.010,0.910200,0.067096,97.443565,2.524971,3,"[0, 1, 2]","[a0_sparse16_flat_p010_s0, a0_sparse16_flat_p0...","[best_test_a0_sparse16_flat_p010_s0.tar, best_...",[full_test_eval.survival_percent]
8,AIB flat,AIB flat local 0.10,0.977308,0.005742,98.393799,0.664686,3,"[0, 1, 2]","[a0_aib_01_flat_local_t010_s0, a0_aib_01_flat_...","[best_test_a0_aib_01_flat_local_t010_s0.tar, b...",[full_test_eval.survival_percent]
9,AIB flat,AIB flat local 0.20,0.979650,0.002457,98.307919,1.182769,3,"[0, 1, 2]","[a0_aib_00_flat_local_t020_s0, a0_aib_00_flat_...","[best_test_a0_aib_00_flat_local_t020_s0.tar, b...",[full_test_eval.survival_percent]


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/a0_selected_action0_survival_tradeoff.html


## Pareto Frontier: Action-0 / Survival Tradeoff

This plot keeps one averaged point per condition and highlights the Pareto frontier for the two objectives we want to maximize: action-0 usage and episodic survival. A condition is Pareto-optimal if no other condition has both higher or equal action-0 usage and higher or equal survival, with at least one strict improvement.


In [28]:
def _mark_pareto_frontier(data, *, x_col, y_col):
    if data.empty:
        out = data.copy()
        out["is_pareto"] = pd.Series(dtype=bool)
        return out

    out = data.copy()
    out[x_col] = pd.to_numeric(out[x_col], errors="coerce")
    out[y_col] = pd.to_numeric(out[y_col], errors="coerce")
    values = out[[x_col, y_col]].to_numpy(dtype=float)
    is_pareto = []
    for idx, (x_value, y_value) in enumerate(values):
        if np.isnan(x_value) or np.isnan(y_value):
            is_pareto.append(False)
            continue
        dominated = np.any(
            (values[:, 0] >= x_value)
            & (values[:, 1] >= y_value)
            & ((values[:, 0] > x_value) | (values[:, 1] > y_value))
        )
        is_pareto.append(not bool(dominated))
    out["is_pareto"] = is_pareto
    return out


if "combined_tradeoff_mean_rows" not in globals() or combined_tradeoff_mean_rows.empty:
    print("Run the checkpoint-aligned tradeoff cell above before computing the Pareto frontier.")
    pareto_tradeoff_rows = pd.DataFrame()
    pareto_frontier_rows = pd.DataFrame()
else:
    pareto_tradeoff_rows = combined_tradeoff_mean_rows.copy()
    pareto_tradeoff_rows["variation"] = pareto_tradeoff_rows["variation"].astype(str)

    # The combined comparison table intentionally reuses comparator runs, such as
    # the baseline or flat local t0.20, in several panels. For the Pareto frontier,
    # keep each unique run variant once so duplicated panel membership cannot
    # create duplicated frontier points.
    pareto_tradeoff_rows["_runs_key"] = pareto_tradeoff_rows["runs"].map(str)
    pareto_tradeoff_rows["_checkpoint_key"] = pareto_tradeoff_rows["checkpoint_files"].map(str)
    pareto_tradeoff_rows = (
        pareto_tradeoff_rows
        .drop_duplicates(
            [
                "variation",
                "_runs_key",
                "_checkpoint_key",
                "mean_action0_fraction",
                "mean_survival_pct",
            ],
            keep="first",
        )
        .drop(columns=["_runs_key", "_checkpoint_key"])
        .reset_index(drop=True)
    )

    pareto_tradeoff_rows = _mark_pareto_frontier(
        pareto_tradeoff_rows,
        x_col="mean_action0_fraction",
        y_col="mean_survival_pct",
    )
    pareto_frontier_rows = (
        pareto_tradeoff_rows[pareto_tradeoff_rows["is_pareto"]]
        .sort_values(["mean_action0_fraction", "mean_survival_pct"], kind="stable")
        .reset_index(drop=True)
    )

    pareto_frontier_table = pareto_frontier_rows[[
        "tradeoff_family",
        "variation",
        "mean_action0_fraction",
        "std_action0_fraction",
        "mean_survival_pct",
        "std_survival_pct",
        "n_seeds",
        "seeds",
        "runs",
        "checkpoint_files",
        "survival_metrics",
    ]].rename(columns={
        "mean_action0_fraction": "pareto_mean_action0_fraction",
        "std_action0_fraction": "pareto_std_action0_fraction",
        "mean_survival_pct": "pareto_mean_survival_pct",
        "std_survival_pct": "pareto_std_survival_pct",
    })
    print(f"Pareto frontier conditions: {len(pareto_frontier_table)} / {len(pareto_tradeoff_rows)}")
    display(pareto_frontier_table)

    fig_pareto_checkpoint_action0_survival = go.Figure()
    for family in family_order:
        for variation in variation_order:
            points = pareto_tradeoff_rows[
                pareto_tradeoff_rows["tradeoff_family"].astype(str).eq(family)
                & pareto_tradeoff_rows["variation"].astype(str).eq(variation)
            ]
            if points.empty:
                continue
            row = points.iloc[0]
            is_pareto = bool(row["is_pareto"])
            color = variation_colors.get(variation, "#999999")
            symbol = family_symbols.get(family, "circle")
            fig_pareto_checkpoint_action0_survival.add_trace(
                go.Scatter(
                    x=[row["mean_action0_fraction"]],
                    y=[row["mean_survival_pct"]],
                    mode="markers",
                    name=f"{family} | {variation}",
                    showlegend=False,
                    marker={
                        "color": color,
                        "symbol": symbol,
                        "size": 11 if is_pareto else 6,
                        "opacity": 0.96 if is_pareto else 0.20,
                        "line": {
                            "color": "black" if is_pareto else "rgba(80,80,80,0.35)",
                            "width": 1.5 if is_pareto else 0.5,
                        },
                    },
                    customdata=np.array([[
                        family,
                        variation,
                        "Pareto" if is_pareto else "dominated",
                        row["n_seeds"],
                        str(row["seeds"]),
                        str(row["runs"]),
                        str(row["checkpoint_files"]),
                        str(row["survival_metrics"]),
                        row["std_action0_fraction"],
                        row["std_survival_pct"],
                    ]], dtype=object),
                    hovertemplate=(
                        "<b>%{customdata[0]} | %{customdata[1]}</b><br>"
                        "status=%{customdata[2]}<br>"
                        "mean action-0=%{x:.4f}<br>"
                        "std action-0=%{customdata[8]:.4f}<br>"
                        "mean survival=%{y:.2f}%<br>"
                        "std survival=%{customdata[9]:.2f}%<br>"
                        "n_seeds=%{customdata[3]}<br>"
                        "seeds=%{customdata[4]}<br>"
                        "runs=%{customdata[5]}<br>"
                        "checkpoints=%{customdata[6]}<br>"
                        "survival metrics=%{customdata[7]}<extra></extra>"
                    ),
                )
            )

    frontier_line = pareto_frontier_rows.drop_duplicates([
        "mean_action0_fraction",
        "mean_survival_pct",
    ]).sort_values(["mean_action0_fraction", "mean_survival_pct"])
    if len(frontier_line) >= 2:
        fig_pareto_checkpoint_action0_survival.add_trace(
            go.Scatter(
                x=frontier_line["mean_action0_fraction"],
                y=frontier_line["mean_survival_pct"],
                mode="lines",
                name="Pareto frontier",
                showlegend=True,
                line={"color": "black", "width": 2.2, "dash": "dash"},
                hoverinfo="skip",
            )
        )

    # Compact legend keys.
    family_legend_labels = {
        "HVG heuristic": "HVG heur.",
        "HVG gate": "HVG gate",
        "Sparse16 flat": "Sparse flat",
        "Sparse16 gated": "Sparse gated",
        "AIB flat": "AIB flat",
        "AIB gate": "AIB gate",
    }
    for family in family_order:
        fig_pareto_checkpoint_action0_survival.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                name=family_legend_labels.get(family, family),
                legendgroup="family_shape",
                legendgrouptitle_text="shape = family",
                marker={
                    "symbol": family_symbols.get(family, "circle"),
                    "color": "#4a4a4a",
                    "size": 8,
                    "line": {"color": "black", "width": 0.7},
                },
                hoverinfo="skip",
            )
        )
    for variation in variation_order:
        fig_pareto_checkpoint_action0_survival.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                name=str(variation),
                legendgroup="variation_color",
                legendgrouptitle_text="color = variation",
                marker={
                    "symbol": "circle",
                    "color": variation_colors.get(variation, "#999999"),
                    "size": 8,
                    "line": {"color": "black", "width": 0.4},
                },
                hoverinfo="skip",
            )
        )
    fig_pareto_checkpoint_action0_survival.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            name="Pareto point",
            legendgroup="pareto_status",
            legendgrouptitle_text="frontier",
            marker={
                "symbol": "circle",
                "color": "white",
                "size": 10,
                "line": {"color": "black", "width": 1.5},
            },
            hoverinfo="skip",
        )
    )
    fig_pareto_checkpoint_action0_survival.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            name="dominated",
            legendgroup="pareto_status",
            marker={
                "symbol": "circle",
                "color": "rgba(120,120,120,0.25)",
                "size": 6,
                "line": {"color": "rgba(80,80,80,0.35)", "width": 0.5},
            },
            hoverinfo="skip",
        )
    )

    fig_pareto_checkpoint_action0_survival.add_annotation(
        text="Pareto frontier: maximize survival and action-0",
        x=0.01,
        y=102,
        xref="x",
        yref="y",
        showarrow=False,
        font={"size": 11, "color": "#333"},
        align="left",
    )
    fig_pareto_checkpoint_action0_survival.add_annotation(
        text="better",
        x=0.98,
        y=102,
        xref="x",
        yref="y",
        showarrow=True,
        ax=-55,
        ay=35,
        font={"size": 11, "color": "#333"},
        arrowcolor="#333",
    )
    fig_pareto_checkpoint_action0_survival.update_layout(
        title=(
            "All A0 families: Pareto frontier of mean checkpoint-aligned action-0 / survival"
            "<br><sup>non-dominated points are highlighted; shape = family; color = variation</sup>"
        ),
        template="plotly_white",
        height=650,
        width=1180,
        hovermode="closest",
        legend={
            "orientation": "v",
            "yanchor": "top",
            "y": 1,
            "xanchor": "left",
            "x": 1.01,
            "font": {"size": 10},
            "itemsizing": "constant",
        },
        margin={"l": 75, "r": 280, "t": 105, "b": 70},
    )
    pareto_x_range = _plot_range_with_padding(
        pareto_tradeoff_rows["mean_action0_fraction"],
        lower=0,
        upper=1,
        min_span=0.08,
    )
    pareto_y_range = _plot_range_with_padding(
        pareto_tradeoff_rows["mean_survival_pct"],
        lower=0,
        upper=105,
        min_span=8,
    )
    fig_pareto_checkpoint_action0_survival.update_xaxes(
        title_text="mean action-0 fraction across agents",
        range=pareto_x_range,
    )
    fig_pareto_checkpoint_action0_survival.update_yaxes(
        title_text="episodic survival (%)",
        range=pareto_y_range,
    )

    if SAVE_FIGURES:
        save_path = ctx["FIG_DIR"] / "a0_all_families_checkpoint_action0_survival_pareto_frontier.html"
        fig_pareto_checkpoint_action0_survival.write_html(save_path, include_plotlyjs="cdn")
        print(f"Saved: {save_path}")
    if SHOW_FIGURES:
        fig_pareto_checkpoint_action0_survival.show()


Pareto frontier conditions: 3 / 17


,tradeoff_family,variation,pareto_mean_action0_fraction,pareto_std_action0_fraction,pareto_mean_survival_pct,pareto_std_survival_pct,n_seeds,seeds,runs,checkpoint_files,survival_metrics
0,AIB flat,flat local t0.20,0.979650,0.002457,100.000000,0.000000,3,"[0, 1, 2]","[a0_aib_00_flat_local_t020_s0, a0_aib_00_flat_...","[best_test_a0_aib_00_flat_local_t020_s0.tar, b...",[test/charts/episodic_survival]
1,AIB flat,flat non-idle t0.20,0.985926,0.006870,99.616815,0.663695,3,"[0, 1, 2]","[a0_aib_04_flat_nonidle_t020_s0, a0_aib_04_fla...","[best_test_a0_aib_04_flat_nonidle_t020_s0.tar,...",[test/charts/episodic_survival]
2,AIB gate,gate h-greedy t0.20,0.999447,0.000505,50.374091,22.056380,3,"[0, 1, 2]","[a0_aib_03_gate_hgreedy_sep_local_t020_s0, a0_...",[best_test_a0_aib_03_gate_hgreedy_sep_local_t0...,[test/charts/episodic_survival]


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/a0_all_families_checkpoint_action0_survival_pareto_frontier.html
